# 2-SLM Solution-Guidance Reasoning Pipeline (SGFT)

**Phase A (Planner)** — Qwen3-4B + SGFT LoRA → emits a semantic plan, *no arithmetic*
**Phase B (Executor)** — Qwen2.5-Math-1.5B, **frozen** → consumes question + plan, does the arithmetic

Zero chat templates. Every prompt is an explicit manual string, defined once in Cell 1 and
read from disk by every other cell, so train/inference skew is structurally impossible.

---

## Run order

| Cell | Does | Typical runtime |
|---|---|---|
| 1 | Install stack, detect GPU, write the prompt contract | 2–4 min |
| 2 | Load + validate `sg_dataset_improv.json`, render exact training strings | < 30 s |
| 3 | Load Qwen3-4B 4-bit, attach LoRA, train 3 epochs, save to Drive, smoke test | 1–3 h on T4 |
| 4 | Load both models, evaluate on GSM8K, stream results to CSV | ~8–15 s/problem |

Cells 3 and 4 each re-read config from disk, so both survive a runtime restart.

---

## Four failure modes this notebook defends against

**1. `pad_token == eos_token`.** Qwen ships both as `<|endoftext|>`. The default causal-LM
collator masks every `pad_token_id` to `-100` — including the one real EOS you appended.
The model then never learns to stop and rambles until it hits the token cap. Cell 3 forces a
distinct pad token that already exists in the vocab, so no embedding resize is needed.

**2. TRL API drift.** Between TRL 0.9 and 0.24, `tokenizer`→`processing_class`,
`max_seq_length`→`max_length`, `evaluation_strategy`→`eval_strategy`, and the args moved into
`SFTConfig`. Cell 3 introspects the installed signatures and adapts instead of pinning versions
that will rot. (Verified against three simulated API generations.)

**3. `packing=True`.** Packing concatenates samples to fill the context window, which destroys
the explicit per-sample string boundaries the whole design rests on. Forced to `False`.

**4. Qwen3 `<think>` blocks.** Qwen3 can emit reasoning blocks even without a chat template.
Cell 4 strips them before the plan reaches Phase B.

---

## On `learning_rate = 2e-5`

Your spec, and what runs by default. The trade-off is worth knowing: catastrophic forgetting is
a *full-finetuning* failure mode, and LoRA at r=16 already constrains drift to ~0.3% of
parameters, so the usual LoRA range is 1e-4 to 2e-4. At 2e-5 the adapter often **underfits the
output format** — the planner writes prose instead of `Step 1:`, or starts doing arithmetic.

The smoke test at the end of Cell 3 is the gate: it needs ≥90% `Step 1:` compliance and ≥90%
arithmetic-free. If it fails, set `learning_rate = 1e-4` and re-run Cell 3. Nothing else changes.

## Cell 1 — Environment setup & the prompt contract

Installs the stack, auto-tunes batch geometry to the detected GPU, and writes
`/content/sgft_project.json`. That file is the single source of truth for the three prompt
templates — no later cell retypes them.

In [1]:
# =============================================================================
# CELL 1 - ENVIRONMENT SETUP, DEPENDENCIES & GLOBAL CONTRACT
# =============================================================================
# This cell does three things:
#   1. Installs the fine-tuning stack (Unsloth + HF ecosystem).
#   2. Detects hardware and auto-derives safe batch sizes.
#   3. Writes ONE canonical config file containing the three prompt templates.
#
# (3) is the single most important part of this project. Train/inference skew
# is almost always caused by the prompt string being re-typed in a second
# place. Every later cell reads the templates from /content/sgft_project.json
# instead of hardcoding them, so the strings physically cannot drift.
# =============================================================================

import os
import sys
import json
import shutil
import platform
import subprocess
import importlib.util

# -----------------------------------------------------------------------------
# 1.1  Small shell helper (no ! magics, so this file stays importable as .py)
# -----------------------------------------------------------------------------
def sh(cmd: str, quiet: bool = False) -> int:
    """Run a shell command, streaming output. Never raises - returns exit code."""
    if not quiet:
        print(f"$ {cmd}")
    try:
        return subprocess.run(cmd, shell=True, check=False).returncode
    except Exception as exc:  # pragma: no cover
        print(f"  [warn] command failed to launch: {exc}")
        return 1


def have(pkg: str) -> bool:
    """True if an importable module exists, without importing it."""
    try:
        return importlib.util.find_spec(pkg) is not None
    except (ImportError, ValueError, ModuleNotFoundError):
        return False


IN_COLAB = ("google.colab" in sys.modules) or any(k.startswith("COLAB_") for k in os.environ)
print(f"Python      : {platform.python_version()}")
print(f"Colab       : {IN_COLAB}")

# WARNING: if transformers is already imported before Unsloth installs/patches,
# Colab needs a runtime restart. We detect and flag it loudly.
TRANSFORMERS_PRELOADED = "transformers" in sys.modules

# -----------------------------------------------------------------------------
# 1.2  Install
# -----------------------------------------------------------------------------
# Strategy: let `unsloth` resolve its own pinned transformers/trl/peft versions
# (it is very opinionated and pinning them ourselves is the #1 cause of
# "SFTTrainer got an unexpected keyword argument" errors). Then upgrade only
# unsloth itself from git with --no-deps so nothing else moves.
INSTALL = True  # set False on re-runs after a restart to save ~2 minutes

if INSTALL:
    sh("pip install -q --upgrade pip")
    if not have("unsloth"):
        sh("pip install -q unsloth unsloth_zoo")
    # Nightly Unsloth: needed for the newest architectures (e.g. Qwen3).
    sh("pip install -q --upgrade --no-cache-dir --no-deps "
       "git+https://github.com/unslothai/unsloth.git "
       "git+https://github.com/unslothai/unsloth-zoo.git")

    # Only install what is genuinely missing - never blanket-upgrade.
    for mod, pipname in [
        ("bitsandbytes", "bitsandbytes"),
        ("accelerate", "accelerate"),
        ("peft", "peft"),
        ("trl", "trl"),
        ("datasets", "datasets"),
        ("sentencepiece", "sentencepiece"),
        ("pandas", "pandas"),
    ]:
        if not have(mod):
            sh(f"pip install -q {pipname}")

# -----------------------------------------------------------------------------
# 1.3  Version + hardware report
# -----------------------------------------------------------------------------
def _ver(mod: str) -> str:
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return "not installed"


print("\n--- Installed versions -------------------------------------------")
for m in ["torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets", "unsloth"]:
    print(f"  {m:<14}: {_ver(m)}")

import torch  # noqa: E402

GPU_NAME, VRAM_GB, SUPPORTS_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    VRAM_GB = round(props.total_memory / 1024 ** 3, 1)
    SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
    print(f"\nGPU           : {GPU_NAME}")
    print(f"VRAM          : {VRAM_GB} GB")
    print(f"Compute cap   : {props.major}.{props.minor}")
    print(f"bfloat16      : {SUPPORTS_BF16}  ({'bf16' if SUPPORTS_BF16 else 'fp16'} will be used)")
else:
    print("\n[FATAL] No CUDA GPU detected. Runtime > Change runtime type > GPU (T4 minimum).")

# Auto-tune batch geometry to the card. Effective batch is held at 16 so the
# learning-rate schedule behaves identically across hardware.
if VRAM_GB >= 38:      # A100 40G / L40S / H100
    MICRO_BS, GRAD_ACC = 8, 2
elif VRAM_GB >= 22:    # A10G / L4 24G / 3090 / 4090
    MICRO_BS, GRAD_ACC = 4, 4
else:                  # T4 16G, the Colab free-tier default
    MICRO_BS, GRAD_ACC = 2, 8
print(f"Batch geometry: micro={MICRO_BS} x accum={GRAD_ACC} -> effective={MICRO_BS * GRAD_ACC}")

if TRANSFORMERS_PRELOADED:
    print("\n" + "!" * 74)
    print("! `transformers` was imported BEFORE Unsloth patched it.")
    print("! Go to Runtime > Restart session, then re-run this cell with INSTALL=False.")
    print("!" * 74)

# -----------------------------------------------------------------------------
# 1.4  THE CONTRACT - canonical prompt templates + paths
# -----------------------------------------------------------------------------
# Note on the deliberate asymmetry between PROMPT_TRAIN and PROMPT_PLANNER:
#   train    : "...\nQuestion: {q}\nPlan:\n{plan}<|endoftext|>"
#   inference: "...\nQuestion: {q}\nPlan:"
# The inference prompt is an EXACT character-level prefix of the training
# string. The model is therefore asked to continue from a state it has seen
# 3,000 times, and it generates the "\n" after "Plan:" itself. Do not "helpfully"
# add a trailing newline to the inference prompt - that would be skew.

PROMPT_TRAIN = (
    "Instruction: Provide a semantic plan without math.\n"
    "Question: {input}\n"
    "Plan:\n"
    "{output}<|endoftext|>"
)

PROMPT_PLANNER = (
    "Instruction: Provide a semantic plan without math.\n"
    "Question: {question}\n"
    "Plan:\n"          # trailing newline: ':\n' is ONE Qwen token, as in training
)

PROMPT_EXECUTOR = (
    "Solve the following math problem. A step-by-step plan is provided; follow "
    "it.\n"
    "Reason step by step, and put your final answer within \\boxed{}.\n\n"
    "Problem: {question}\n\n"
    "Plan:\n{plan}\n\n"
    "Solution:"
)

DRIVE_ROOT = "/content/drive/MyDrive/SGFT_2SLM"
LOCAL_ROOT = "/content/sgft"

PROJECT = {
    # --- prompt contract (single source of truth) ---
    "prompt_train": PROMPT_TRAIN,
    "prompt_planner": PROMPT_PLANNER,
    "prompt_executor": PROMPT_EXECUTOR,
    "eos_literal": "<|endoftext|>",

    # --- models ---
    # Candidates are tried in order; first one that loads wins. Unsloth's
    # pre-quantised repos download ~4x faster than quantising Qwen/Qwen3-4B live.
    "planner_base_candidates": [
        "unsloth/Qwen3-4B-unsloth-bnb-4bit",
        "unsloth/Qwen3-4B",
        "Qwen/Qwen3-4B",
    ],
    # Swap to this if you are on a T4 and 4B training is too slow (~3h/3 epochs):
    #   ["unsloth/Qwen2.5-Math-1.5B-bnb-4bit", "Qwen/Qwen2.5-Math-1.5B"]
    "executor_model": "Qwen/Qwen2.5-Math-1.5B",

    # --- data ---
    "dataset_filename": "sg_dataset_improv.json",
    "dataset_search_paths": [
        "/content/sg_dataset_improv.json",
        "/content/data/sg_dataset_improv.json",
        f"{DRIVE_ROOT}/data/sg_dataset_improv.json",
        "/content/drive/MyDrive/sg_dataset_improv.json",
        "/mnt/user-data/uploads/sg_dataset_improv.json",
    ],
    "val_ratio": 0.05,          # 3,000 samples -> ~150 val. 15% would waste 450.
    "min_steps": 2,
    "max_steps": 6,
    "drop_plans_with_math": True,

    # --- training ---
    "max_seq_length": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "learning_rate": 2e-5,      # per spec. See the LR note printed below.
    "epochs": 3,
    "lr_scheduler": "cosine",
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "micro_batch_size": MICRO_BS,
    "grad_accum": GRAD_ACC,
    "seed": 3407,
    "train_on_responses_only": False,  # see Cell 3 for why this is usually a win

    # --- paths ---
    "local_root": LOCAL_ROOT,
    "drive_root": DRIVE_ROOT,
    "processed_dataset": f"{LOCAL_ROOT}/data/sg_processed.jsonl",
    "adapter_dir": f"{LOCAL_ROOT}/adapters/phaseA_planner",
    "drive_adapter_dir": f"{DRIVE_ROOT}/adapters/phaseA_planner",
    "results_csv": f"{LOCAL_ROOT}/results/pipeline_detailed_results.csv",
    "drive_results_csv": f"{DRIVE_ROOT}/results/pipeline_detailed_results.csv",

    # --- inference ---
    "planner_max_new_tokens": 192,
    "executor_max_new_tokens": 640,
    "temperature": 0.0,          # 0.0 => greedy. Reproducible evals only.
    "repetition_penalty": 1.15,

    # --- runtime facts (informational) ---
    "gpu_name": GPU_NAME,
    "vram_gb": VRAM_GB,
    "supports_bf16": SUPPORTS_BF16,
}

for sub in ["data", "adapters", "results", "logs"]:
    os.makedirs(os.path.join(LOCAL_ROOT, sub), exist_ok=True)

CONFIG_PATH = "/content/sgft_project.json" if IN_COLAB else "./sgft_project.json"
with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
    json.dump(PROJECT, fh, indent=2)

print(f"\nConfig written -> {CONFIG_PATH}")
print("\n--- Prompt contract (repr, so whitespace is visible) --------------")
print("TRAIN    :", repr(PROMPT_TRAIN))
print("PLANNER  :", repr(PROMPT_PLANNER))
print("EXECUTOR :", repr(PROMPT_EXECUTOR))

# -----------------------------------------------------------------------------
# 1.5  Engineering note you should read before training
# -----------------------------------------------------------------------------
print("""
--- STATUS ------------------------------------------------------------------
Phase A (planner) is trained and validated: 100% 'Step N [OP]:' format, 100%
operators inside the closed vocabulary, 0% arithmetic leakage on 150 problems.
lr=2e-5 was sufficient. Do not retrain.

Cell 3's smoke test gates on three things: tagged format, arithmetic-free, and
operators in vocabulary. If a future re-run drops below 90% on any of them,
raise learning_rate to 1e-4 and set train_on_responses_only=True.

The open problem is Phase B, not Phase A. On the last run the executor never
emitted #### (2/150) and reached \\boxed only 35/150, leaving 61% of scoring to
the equals_preramble heuristic. That is what the \\boxed{} prompt is meant to
fix.
-----------------------------------------------------------------------------""")

Python      : 3.13.15
Colab       : True
$ pip install -q --upgrade pip
$ pip install -q unsloth unsloth_zoo
$ pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git git+https://github.com/unslothai/unsloth-zoo.git

--- Installed versions -------------------------------------------
  torch         : 2.11.0+cu128
  transformers  : 5.5.0
  trl           : 0.24.0


  peft          : 0.20.0
  bitsandbytes  : 0.50.2
  accelerate    : 1.14.0
  datasets      : 4.3.0


/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1556: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  unsloth       : 2026.9.5

GPU           : Tesla T4
VRAM          : 14.6 GB
Compute cap   : 7.5
bfloat16      : False  (fp16 will be used)
Batch geometry: micro=2 x accum=8 -> effective=16

Config written -> /content/sgft_project.json

--- Prompt contract (repr, so whitespace is visible) --------------
TRAIN    : 'Instruction: Provide a semantic plan without math.\nQuestion: {input}\nPlan:\n{output}<|endoftext|>'
PLANNER  : 'Instruction: Provide a semantic plan without math.\nQuestion: {question}\nPlan:\n'
EXECUTOR : 'Solve the following math problem. A step-by-step plan is provided; follow it.\nReason step by step, and put your final answer within \\boxed{}.\n\nProblem: {question}\n\nPlan:\n{plan}\n\nSolution:'

--- STATUS ------------------------------------------------------------------
Phase A (planner) is trained and validated: 100% 'Step N [OP

In [2]:
# =============================================================================
# CELL 0 - SESSION RECOVERY  (run after Cell 1, before Cell 4)
# =============================================================================
# Colab recycled the VM, so /content is empty. What survives on Drive:
#   adapters/phaseA_planner/   the LoRA + tokenizer + run_manifest.json
#   results/pipeline_detailed_results.csv
#
# That is everything Cell 4 needs. Cells 2 and 3 stay skipped - no retraining.
#
# This cell mounts Drive, verifies the artifacts are intact, copies the adapter
# back to local disk (Drive I/O is slow enough to matter during generation),
# and restores the results CSV so the re-score can run.
# =============================================================================

import os
import json
import shutil

from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CONFIG_PATH = "/content/sgft_project.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    PROJECT = json.load(fh)

DRIVE_ADAPTER = PROJECT["drive_adapter_dir"]
LOCAL_ADAPTER = PROJECT["adapter_dir"]
DRIVE_CSV = PROJECT["drive_results_csv"]
LOCAL_CSV = PROJECT["results_csv"]

# -----------------------------------------------------------------------------
# 0.1  Verify the adapter survived and is complete
# -----------------------------------------------------------------------------
REQUIRED = ["adapter_config.json", "adapter_model.safetensors", "run_manifest.json"]

print(f"[recover] checking {DRIVE_ADAPTER}")
if not os.path.isdir(DRIVE_ADAPTER):
    raise FileNotFoundError(
        f"Adapter directory missing on Drive: {DRIVE_ADAPTER}\n"
        f"Check MyDrive/SGFT_2SLM/adapters/ - if only 'phaseA_planner_prev' exists, "
        f"rename it. If nothing is there, the adapter is gone and Cell 3 must be "
        f"re-run (~1-3h)."
    )

present = os.listdir(DRIVE_ADAPTER)
missing = [f for f in REQUIRED if f not in present]
for f in sorted(present):
    size = os.path.getsize(os.path.join(DRIVE_ADAPTER, f)) / 1024 ** 2
    print(f"    {f:<34} {size:>8.1f} MB")
if missing:
    raise FileNotFoundError(f"Adapter is incomplete, missing: {missing}")
print("[recover] adapter OK")

# -----------------------------------------------------------------------------
# 0.2  Confirm the manifest carries the post-fix planner prompt
# -----------------------------------------------------------------------------
with open(os.path.join(DRIVE_ADAPTER, "run_manifest.json"), "r", encoding="utf-8") as fh:
    manifest = json.load(fh)

print(f"\n[recover] manifest")
print(f"    base_model        : {manifest.get('base_model')}")
print(f"    prompt_planner    : {manifest.get('prompt_planner')!r}")
print(f"    token_verified    : {manifest.get('prompt_planner_token_verified', False)}")
print(f"    operator_vocab    : {manifest.get('operator_vocab')}")
print(f"    max_steps         : {manifest.get('max_steps')}")
print(f"    n_train / n_val   : {manifest.get('n_train')} / {manifest.get('n_val')}")

# Not fatal either way: section 4.0.1 derives the planner prompt from
# prompt_train rather than trusting a stored one, precisely so a lost or stale
# config cannot silently reintroduce the ':' vs ':\n' boundary bug.
if not manifest.get("prompt_planner", "").endswith("\n"):
    print("    [note] stored planner prompt lacks the trailing newline - "
          "section 4.0.1 will repair it automatically at load time.")

# -----------------------------------------------------------------------------
# 0.3  Copy the adapter back to local disk
# -----------------------------------------------------------------------------
# Generation reads adapter weights repeatedly; serving them over the Drive FUSE
# mount adds noticeable latency per call.
os.makedirs(os.path.dirname(LOCAL_ADAPTER), exist_ok=True)
if os.path.exists(LOCAL_ADAPTER):
    shutil.rmtree(LOCAL_ADAPTER)
shutil.copytree(DRIVE_ADAPTER, LOCAL_ADAPTER)
print(f"\n[recover] adapter -> {LOCAL_ADAPTER}")

# -----------------------------------------------------------------------------
# 0.4  Restore the results CSV so the Fix-A re-score has something to read
# -----------------------------------------------------------------------------
os.makedirs(os.path.dirname(LOCAL_CSV), exist_ok=True)
if os.path.exists(DRIVE_CSV):
    shutil.copy2(DRIVE_CSV, LOCAL_CSV)
    with open(LOCAL_CSV, "r", encoding="utf-8") as fh:
        n_rows = sum(1 for _ in fh) - 1
    print(f"[recover] results CSV -> {LOCAL_CSV}  ({n_rows} rows)")
    print("[recover] run the Fix-A re-score cell next to recover your true accuracy")
else:
    print(f"[recover] no results CSV on Drive at {DRIVE_CSV}")
    print("[recover] the re-score has nothing to read - skip it and go straight to Cell 4")

# -----------------------------------------------------------------------------
# 0.5  Apply the Change-2 setting that Cell 1 may not carry
# -----------------------------------------------------------------------------
# Harmless if Cell 1 already has it; required if you skipped editing Cell 1.
PROJECT["repetition_penalty"] = 1.15
with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
    json.dump(PROJECT, fh, indent=2)
print(f"\n[recover] repetition_penalty = {PROJECT['repetition_penalty']}")
print(f"[recover] config written -> {CONFIG_PATH}")
print("\n[recover] ready. Next: Fix-A re-score cell, then Cell 4. Skip Cells 2, 3, 3.5.")

Mounted at /content/drive
[recover] checking /content/drive/MyDrive/SGFT_2SLM/adapters/phaseA_planner
    README.md                               0.0 MB
    adapter_config.json                     0.0 MB
    adapter_model.safetensors             126.1 MB
    chat_template.jinja                     0.0 MB
    run_manifest.json                       0.0 MB
    tokenizer.json                         10.9 MB
    tokenizer_config.json                   0.0 MB
[recover] adapter OK

[recover] manifest
    base_model        : unsloth/Qwen3-4B-unsloth-bnb-4bit
    prompt_planner    : 'Instruction: Provide a semantic plan without math.\nQuestion: {question}\nPlan:\n'
    token_verified    : True
    operator_vocab    : ['ADD', 'COMPARE', 'CONVERT', 'COUNT', 'DIVIDE', 'MULTIPLY', 'RATIO', 'SUBTRACT']
    max_steps         : 9
    n_train / n_val   : 2850 / 150

[recover] adapter -> /content/sgft/adapters/phaseA_planner
[recover] results CSV -> /content/sgft/results/pipeline_detailed_results.csv  

## Cell 2 — Dataset processing

Loads `sg_dataset_improv.json` (tolerant of `input`/`question` and `output`/`plan` key
spellings, JSON or JSONL), cleans and validates every plan, deduplicates, and renders the exact
training string:

```
Instruction: Provide a semantic plan without math.\nQuestion: {input}\nPlan:\n{output}<|endoftext|>
```

Rejection reasons are counted and printed, so you can see exactly what your teacher model got
wrong. The cell ends with hard assertions that the planner inference prompt is a byte-for-byte
prefix of the training string — if that fails, stop and fix it before spending GPU hours.

In [ ]:
# =============================================================================
# CELL 2 - DATASET PROCESSING PIPELINE  (operator-tagged SG format)
# =============================================================================
# Actual format of sg_dataset_improv.json (7,473 rows, all {"input","output"}):
#
#   "Step 1 [DIVIDE]: Determine the number of clips sold in May based on April.
#
#    Step 2 [ADD]: Determine the total number of clips sold altogether."
#
# Two things differ from the plain SGFT format and both matter:
#   1. Every step carries an [OPERATOR] tag. That is strictly more information
#      than the base paper's format - it tells Phase B *which* operation to
#      apply, not just what to think about. We keep it, canonicalise it, and
#      later validate planner output against the closed operator vocabulary.
#   2. Steps are separated by BLANK lines (7,151 rows) or single newlines (322).
#      We normalise to exactly one step per line so the training target has an
#      identical shape across all samples.
#
# Audit of the raw file (this is why the validators are tuned as they are):
#   - 0 malformed rows, 0 stray text outside steps, 0 non-monotonic numbering,
#     0 duplicate questions, 0 arithmetic leaks. The file is genuinely clean.
#   - Step counts run 1..9 (mode 3). Cell 1 guessed max_steps=6, which would
#     have silently discarded 397 of the hardest multi-step problems.
#   - 20 step descriptions contain a digit, all legitimate entity references
#     ("30-day pass", "2-point shots", "100-meter backstroke"). The arithmetic
#     validator must not flag these, so it only matches digit-operator-digit.
#   - Operator vocabulary is 13, but 5 are near-singletons (EQUATE 2, EQUAL 2,
#     SOLVE 1, ASSIGN 1, EXPRESS 1). Those rows are dropped by a frequency
#     floor so the planner learns a closed 8-symbol vocabulary that can be
#     validated at inference time.
# =============================================================================

import os
import re
import json
import random
import hashlib
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

from datasets import Dataset

# -----------------------------------------------------------------------------
# 2.1  Load the contract from Cell 1, then override the guesses it made about
#      the data before the data had been seen. Cell 1 needs no edit.
# -----------------------------------------------------------------------------
CONFIG_PATH = "/content/sgft_project.json" if os.path.exists("/content/sgft_project.json") else "./sgft_project.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    PROJECT = json.load(fh)

PROJECT["min_steps"] = 2            # 3 single-step rows exist; a 1-step "plan" teaches nothing
PROJECT["max_steps"] = 9            # keep the 7-9 step problems - that is where planning pays
PROJECT["min_operator_freq"] = 10   # drops the 5 singleton operators
# Base-paper data-efficiency result: 3,000 SG samples beat 30,000 CoT samples.
# Set to None to train on all ~7,460 valid rows (roughly 2.5x the runtime).
PROJECT["max_train_samples"] = 3000
PROJECT["stratify_by_step_count"] = True

random.seed(PROJECT["seed"])


# -----------------------------------------------------------------------------
# 2.2  File discovery
# -----------------------------------------------------------------------------
def locate_dataset(project: Dict[str, Any]) -> str:
    """Find sg_dataset_improv.json across the usual Colab/Drive/upload paths."""
    for path in project["dataset_search_paths"]:
        if os.path.exists(path):
            print(f"[data] found -> {path}")
            return path

    target = project["dataset_filename"]
    for root in ["/content", "/mnt/user-data/uploads", "."]:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count(os.sep) > 6:
                dirnames[:] = []
                continue
            if target in filenames:
                found = os.path.join(dirpath, target)
                print(f"[data] found via scan -> {found}")
                return found

    raise FileNotFoundError(
        f"Could not locate '{target}'. Upload it to /content/ or mount Drive.\n"
        f"Searched: {project['dataset_search_paths']}"
    )


def read_records(path: str) -> List[Dict[str, Any]]:
    """Read .json (list or {'data': [...]}) or .jsonl transparently."""
    with open(path, "r", encoding="utf-8") as fh:
        head = fh.read(2048)
        fh.seek(0)
        stripped = head.lstrip()

        if stripped.startswith("["):
            data = json.load(fh)
        elif stripped.startswith("{") and '"data"' in head:
            blob = json.load(fh)
            data = blob.get("data") or blob.get("samples") or blob.get("records")
        elif stripped.startswith("{"):
            fh.seek(0)
            data = []
            for line_no, line in enumerate(fh, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError:
                    if line_no == 1:
                        fh.seek(0)
                        data = json.load(fh)
                        if isinstance(data, dict):
                            data = [data]
                        break
                    raise
        else:
            raise ValueError(f"Unrecognised file format at {path}")

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records, got {type(data)}")
    print(f"[data] {len(data)} raw records loaded")
    return data


# -----------------------------------------------------------------------------
# 2.3  Field normalisation
# -----------------------------------------------------------------------------
Q_KEYS = ("input", "question", "problem", "query", "prompt")
P_KEYS = ("output", "plan", "guidance", "solution_guidance", "sg", "response")


def normalise(rec: Dict[str, Any]) -> Optional[Tuple[str, str]]:
    """Return (question, plan) or None if the record is unusable."""
    q = p = None
    for k in Q_KEYS:
        if isinstance(rec.get(k), str) and rec[k].strip():
            q = rec[k]
            break
    for k in P_KEYS:
        if isinstance(rec.get(k), str) and rec[k].strip():
            p = rec[k]
            break
    if q is None or p is None:
        return None
    # GSM8K questions contain stray double spaces; collapse so the rendered
    # prompt is stable regardless of source whitespace noise.
    return " ".join(q.split()), p.strip()


# -----------------------------------------------------------------------------
# 2.4  Step parsing for the "Step N [OP]: desc" format
# -----------------------------------------------------------------------------
# The operator tag is OPTIONAL at parse time, so the same parser can clean
# planner output at inference where the model may drop it.
STEP_RE = re.compile(
    r"^[ \t]*Step[ \t]*(\d+)[ \t]*(?:\[[ \t]*([^\]]{1,32}?)[ \t]*\])?[ \t]*:[ \t]*(.*?)[ \t]*$",
    re.MULTILINE | re.IGNORECASE,
)
STEP_SPLIT_RE = re.compile(r"[ \t]*(Step[ \t]*\d+[ \t]*(?:\[|:))", re.IGNORECASE)
THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)

# Arithmetic leakage. Requires digit-operator-digit or '=' followed by a number,
# so "30-day pass" and "2-point shots" stay legal. Bare '-' is only treated as
# subtraction when whitespace-delimited ("12 - 5").
ARITH_RE = re.compile(
    r"\d[ \t]*(?:[+*/×÷]|\*\*)[ \t]*\d"
    r"|\d[ \t]+-[ \t]+\d"
    r"|\d[ \t]*[xX][ \t]*\d"
    r"|=[ \t]*\$?\d"
    r"|<<[^>]*>>"
)


def parse_steps(text: str) -> List[Tuple[int, str, str]]:
    """Extract [(number, operator_or_empty, description), ...] from a plan."""
    body = THINK_RE.sub("", text or "")
    body = re.sub(r"^\s*(?:Plan|Solution Guidance|Guidance)\s*:\s*", "", body, flags=re.IGNORECASE)
    body = STEP_SPLIT_RE.sub(r"\n\1", body)          # force one step per line
    steps = []
    for num, op, desc in STEP_RE.findall(body):
        desc = " ".join(desc.split())
        if not desc:
            continue
        steps.append((int(num), (op or "").strip().upper(), desc))
    return steps


def render_plan(steps: List[Tuple[int, str, str]]) -> str:
    """
    Canonical target: one step per line, renumbered 1..N, single-newline
    separated, uppercase tag. Collapsing the source blank lines saves ~15% of
    target tokens and makes every sample structurally identical.
    """
    out = []
    for i, (_num, op, desc) in enumerate(steps, 1):
        out.append(f"Step {i} [{op}]: {desc}" if op else f"Step {i}: {desc}")
    return "\n".join(out)


def validate(steps: List[Tuple[int, str, str]], plan: str,
             allowed_ops: Optional[set], project: Dict[str, Any]) -> Optional[str]:
    """Return a rejection reason, or None if the sample is good."""
    if not steps:
        return "no_step_marker"
    n = len(steps)
    if n < project["min_steps"]:
        return f"too_few_steps({n})"
    if n > project["max_steps"]:
        return f"too_many_steps({n})"
    if any(not op for _, op, _ in steps):
        return "missing_operator_tag"
    if allowed_ops is not None:
        rare = {op for _, op, _ in steps if op not in allowed_ops}
        if rare:
            return f"rare_operator({'/'.join(sorted(rare))})"
    if ARITH_RE.search(plan):
        return "arithmetic_leak"
    if len(plan) < 40:
        return "too_short"
    if len(plan) > 2000:
        return "too_long"
    return None


def qhash(question: str) -> str:
    """Stable dedup hash - case/punctuation/whitespace insensitive."""
    return hashlib.md5(re.sub(r"[^a-z0-9]+", "", question.lower()).encode("utf-8")).hexdigest()


def render_training_text(question: str, plan: str, template: str) -> str:
    """
    Apply the frozen template with .replace, not .format - a word problem can
    legitimately contain a brace and .format would raise on it.
    """
    return template.replace("{input}", question).replace("{output}", plan)


# -----------------------------------------------------------------------------
# 2.5  Stratified subsampling
# -----------------------------------------------------------------------------
def stratified_sample(rows: List[Dict[str, Any]], n_target: int, seed: int,
                      key: str = "n_steps") -> List[Dict[str, Any]]:
    """
    Take n_target rows while preserving the step-count distribution. A plain
    shuffle-and-slice drifts the 7-9 step tail, which is exactly the population
    the two-stage pipeline is supposed to help with.
    """
    if n_target >= len(rows):
        return rows

    buckets: Dict[Any, List[Dict[str, Any]]] = defaultdict(list)
    for row in rows:
        buckets[row[key]].append(row)

    rng = random.Random(seed)
    for bucket in buckets.values():
        rng.shuffle(bucket)

    quotas = {k: int(len(v) * n_target / len(rows)) for k, v in buckets.items()}
    picked: List[Dict[str, Any]] = []
    for k, quota in quotas.items():
        picked.extend(buckets[k][:quota])

    # Hand the rounding remainder to the largest buckets, deterministically.
    pools = [buckets[k][quotas[k]:] for k in sorted(buckets, key=lambda x: -len(buckets[x]))]
    i = 0
    while len(picked) < n_target and any(pools):
        pool = pools[i % len(pools)]
        if pool:
            picked.append(pool.pop(0))
        i += 1

    rng.shuffle(picked)
    return picked[:n_target]


# -----------------------------------------------------------------------------
# 2.6  Build
# -----------------------------------------------------------------------------
def build_dataset(project: Dict[str, Any]) -> Tuple[Dataset, Dataset, Dict[str, Any]]:
    path = locate_dataset(project)
    raw = read_records(path)

    # --- Pass 1: parse everything, learn the operator distribution -----------
    parsed: List[Tuple[str, List[Tuple[int, str, str]], str]] = []
    op_counts: Counter = Counter()
    pre_reasons: Counter = Counter()

    for rec in raw:
        pair = normalise(rec)
        if pair is None:
            pre_reasons["missing_fields"] += 1
            continue
        question, raw_plan = pair
        steps = parse_steps(raw_plan)
        if not steps:
            pre_reasons["no_step_marker"] += 1
            continue
        parsed.append((question, steps, render_plan(steps)))
        op_counts.update(op for _, op, _ in steps if op)

    floor = project["min_operator_freq"]
    allowed_ops = {op for op, c in op_counts.items() if c >= floor}

    print("\n--- Operator vocabulary ------------------------------------------")
    total_ops = max(sum(op_counts.values()), 1)
    for op, count in op_counts.most_common():
        flag = "" if op in allowed_ops else f"   <- below freq floor {floor}, rows dropped"
        print(f"  {op:<12} {count:>6}  {count / total_ops:>6.2%}{flag}")
    print(f"  closed vocabulary : {sorted(allowed_ops)}")

    # --- Pass 2: validate, dedupe, render ------------------------------------
    kept: List[Dict[str, Any]] = []
    reasons: Counter = Counter(pre_reasons)
    seen: set = set()

    for question, steps, plan in parsed:
        h = qhash(question)
        if h in seen:
            reasons["duplicate_question"] += 1
            continue
        why = validate(steps, plan, allowed_ops, project)
        if why:
            reasons[why] += 1
            continue
        seen.add(h)
        kept.append({
            "question": question,
            "plan": plan,
            "n_steps": len(steps),
            "operators": " ".join(op for _, op, _ in steps),
            "text": render_training_text(question, plan, project["prompt_train"]),
        })

    print("\n--- Validation report --------------------------------------------")
    print(f"  raw records      : {len(raw)}")
    print(f"  valid            : {len(kept)}  ({len(kept) / max(len(raw), 1):.1%})")
    if reasons:
        print("  rejected:")
        for why, count in reasons.most_common():
            print(f"      {why:<34} {count}")
    if not kept:
        raise RuntimeError("Every record was rejected. Inspect the reasons above.")

    full_dist = Counter(r["n_steps"] for r in kept)
    n_valid = len(kept)

    # --- Subsample -----------------------------------------------------------
    n_target = project.get("max_train_samples")
    if n_target and n_target < len(kept):
        if project.get("stratify_by_step_count", True):
            kept = stratified_sample(kept, n_target, project["seed"])
            print(f"\n  stratified subsample -> {len(kept)} of {n_valid} "
                  f"(step-count distribution preserved)")
        else:
            random.Random(project["seed"]).shuffle(kept)
            kept = kept[:n_target]
            print(f"\n  random subsample -> {len(kept)} of {n_valid}")

    dist = Counter(r["n_steps"] for r in kept)
    print("\n  step-count distribution (selected vs full valid set):")
    for k in sorted(full_dist):
        sel, tot = dist.get(k, 0), full_dist[k]
        bar = "#" * int((sel / max(len(kept), 1)) * 40)
        print(f"      {k} steps : {sel:>5} / {tot:<5}  sel={sel / max(len(kept), 1):>6.1%} "
              f"full={tot / n_valid:>6.1%} {bar}")

    op_sel = Counter()
    for r in kept:
        op_sel.update(r["operators"].split())
    print("\n  operator mix in the selected set:")
    for op, c in op_sel.most_common():
        print(f"      {op:<12} {c:>6}  {c / max(sum(op_sel.values()), 1):>6.2%}")

    char_lens = sorted(len(r["text"]) for r in kept)
    print(f"\n  training text (chars): min={char_lens[0]}  "
          f"p50={char_lens[len(char_lens) // 2]}  "
          f"p95={char_lens[int(len(char_lens) * 0.95)]}  max={char_lens[-1]}"
          f"   (~{char_lens[-1] // 3} tokens worst case vs "
          f"max_seq_length={project['max_seq_length']})")

    # --- Split. The label is persisted so Cell 3 can restore the exact same
    #     partition after a runtime restart instead of re-shuffling. ----------
    random.Random(project["seed"]).shuffle(kept)
    n_val = max(1, int(len(kept) * project["val_ratio"]))
    for i, row in enumerate(kept):
        row["split"] = "val" if i < n_val else "train"
    val_rows = [r for r in kept if r["split"] == "val"]
    train_rows = [r for r in kept if r["split"] == "train"]
    print(f"\n  split            : {len(train_rows)} train / {len(val_rows)} val")

    os.makedirs(os.path.dirname(project["processed_dataset"]), exist_ok=True)
    with open(project["processed_dataset"], "w", encoding="utf-8") as fh:
        for row in kept:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"  persisted        -> {project['processed_dataset']}")

    stats = {
        "n_raw": len(raw),
        "n_valid": n_valid,
        "n_selected": len(kept),
        "rejections": dict(reasons),
        "step_distribution": dict(dist),
        "operator_counts": dict(op_counts),
        "operator_vocab": sorted(allowed_ops),
    }
    return Dataset.from_list(train_rows), Dataset.from_list(val_rows), stats


train_dataset, val_dataset, data_stats = build_dataset(PROJECT)

# Persist the closed operator vocabulary: Cell 3 records it in the run manifest
# and Cell 4 validates planner output against it.
PROJECT["operator_vocab"] = data_stats["operator_vocab"]
with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
    json.dump(PROJECT, fh, indent=2)


# -----------------------------------------------------------------------------
# 2.7  Prove the prompt contract holds
# -----------------------------------------------------------------------------
print("\n--- Rendered sample #0 (repr - inspect every space and newline) -----")
print(repr(train_dataset[0]["text"]))
print("\n--- Rendered sample #0 (human readable) ----------------------------")
print(train_dataset[0]["text"])
print("\n--- Rendered sample #1 ---------------------------------------------")
print(train_dataset[1]["text"])

_probe = train_dataset[0]["text"]
assert _probe.startswith("Instruction: Provide a semantic plan without math.\nQuestion: "), \
    "Training prefix does not match the planner inference prefix -> guaranteed skew."
assert "\nPlan:\nStep 1 [" in _probe, "Plan block does not open with an operator-tagged Step 1."
assert _probe.endswith(PROJECT["eos_literal"]), \
    "Missing terminal <|endoftext|> -> the model will never learn to stop."
assert "\n\n" not in _probe.split("\nPlan:\n", 1)[1], \
    "A blank line survived inside the plan target - step separators were not normalised."

_rendered_planner = PROJECT["prompt_planner"].replace("{question}", train_dataset[0]["question"])
assert _probe.startswith(_rendered_planner), (
    "Planner inference prompt is NOT a byte-for-byte prefix of the training string.\n"
    f"  inference tail: {_rendered_planner[-70:]!r}\n"
    f"  training  same: {_probe[:len(_rendered_planner)][-70:]!r}"
)
print("\n[OK] Contract assertions passed: planner inference prompt is a literal prefix of "
      "the training string, plans are single-newline separated, every sample ends in exactly "
      "one <|endoftext|>.")

[data] found -> /content/sg_dataset_improv.json
[data] 7473 raw records loaded

--- Operator vocabulary ------------------------------------------
  MULTIPLY       9838  38.72%
  ADD            6226  24.51%
  SUBTRACT       4591  18.07%
  DIVIDE         3751  14.76%
  CONVERT         460   1.81%
  RATIO           417   1.64%
  COMPARE          69   0.27%
  COUNT            47   0.18%
  EQUATE            2   0.01%   <- below freq floor 10, rows dropped
  EQUAL             2   0.01%   <- below freq floor 10, rows dropped
  SOLVE             1   0.00%   <- below freq floor 10, rows dropped
  ASSIGN            1   0.00%   <- below freq floor 10, rows dropped
  EXPRESS           1   0.00%   <- below freq floor 10, rows dropped
  closed vocabulary : ['ADD', 'COMPARE', 'CONVERT', 'COUNT', 'DIVIDE', 'MULTIPLY', 'RATIO', 'SUBTRACT']

--- Validation report --------------------------------------------
  raw records      : 7473
  valid            : 7464  (99.9%)
  rejected:
      too_few_steps(1) 

## Cell 3 — Phase A planner fine-tuning

Unsloth + Qwen3-4B in 4-bit, LoRA r=16 / α=32 on all seven linear projections, cosine schedule,
3 epochs. Saves the adapter, the tokenizer (with the corrected pad/EOS setup) and a
`run_manifest.json` recording exactly what was trained — Cell 4 reads that manifest rather than
guessing.

Ends with the smoke-test acceptance gate. **Read its verdict before running Cell 4.**

In [ ]:
# =============================================================================
# CELL 3 - PHASE A PLANNER: MODEL LOADING & SGFT FINE-TUNING
# =============================================================================
# Four non-obvious things this cell handles, each of which silently ruins the
# run if you skip it:
#
#  (1) PAD == EOS.  Qwen ships pad_token == eos_token == <|endoftext|>. The
#      default causal-LM collator masks every pad_token_id to -100, which also
#      masks the ONE real <|endoftext|> appended in Cell 2. The model then never
#      learns to stop and rambles to the token cap. We force a distinct pad
#      token that already exists in the vocab (no embedding resize).
#
#  (2) TRUNCATION KILLS EOS.  If a sample exceeds max_seq_length, the tail is
#      cut - and the tail is exactly where <|endoftext|> lives. A handful of
#      truncated samples teaches the model that stopping is optional. Section
#      3.5 audits real token lengths and refuses to proceed silently.
#
#  (3) TRL API drift.  Across TRL 0.9 -> 0.24 the SFTTrainer signature changed
#      three times (tokenizer -> processing_class, max_seq_length -> max_length,
#      evaluation_strategy -> eval_strategy, args moved into SFTConfig). We
#      introspect the installed signatures and adapt rather than pin and rot.
#
#  (4) packing MUST be False.  Packing concatenates samples to fill the context
#      window, destroying the explicit per-sample string boundaries.
# =============================================================================

# Unsloth MUST be imported before transformers/trl so its kernels patch in.
from unsloth import FastLanguageModel  # noqa: F401  (import order matters)

import os
import re
import gc
import json
import time
import shutil
import inspect
from typing import Any, Callable, Dict, List, Optional, Set

import torch
from transformers import TrainingArguments
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:  # very old TRL
    SFTConfig = None

CONFIG_PATH = "/content/sgft_project.json" if os.path.exists("/content/sgft_project.json") else "./sgft_project.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    PROJECT = json.load(fh)


# -----------------------------------------------------------------------------
# 3.1  Signature-introspection helpers (the TRL/transformers version shim)
# -----------------------------------------------------------------------------
def accepted_params(target: Callable) -> Set[str]:
    """Parameter names a callable / dataclass __init__ will accept."""
    try:
        return set(inspect.signature(target).parameters.keys())
    except (TypeError, ValueError):
        return set()


def first_supported(accepted: Set[str], *names: str) -> Optional[str]:
    """Return the first alias the target actually supports."""
    for name in names:
        if name in accepted:
            return name
    return None


def filter_kwargs(accepted: Set[str], kwargs: Dict[str, Any], label: str) -> Dict[str, Any]:
    """Drop unsupported kwargs loudly instead of crashing on a version bump."""
    if not accepted:                      # introspection failed - pass through
        return dict(kwargs)
    supported = {k: v for k, v in kwargs.items() if k in accepted}
    dropped = sorted(set(kwargs) - set(supported))
    if dropped:
        print(f"  [shim] {label}: dropped unsupported kwargs {dropped}")
    return supported


# -----------------------------------------------------------------------------
# 3.2  Base model loading with candidate fallback
# -----------------------------------------------------------------------------
def load_base_model(project: Dict[str, Any]):
    """Try each candidate repo in order; return (model, tokenizer, repo_id)."""
    load_accepted = accepted_params(FastLanguageModel.from_pretrained)
    last_error = None

    for repo in project["planner_base_candidates"]:
        print(f"\n[load] attempting {repo} ...")
        kwargs = {
            "model_name": repo,
            "max_seq_length": project["max_seq_length"],
            "dtype": None,               # None => auto (bf16 on Ampere+, fp16 on T4)
            "load_in_4bit": True,
            "full_finetuning": False,
        }
        kwargs = {k: v for k, v in kwargs.items() if not load_accepted or k in load_accepted}
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(**kwargs)
            print(f"[load] OK -> {repo}")
            return model, tokenizer, repo
        except Exception as exc:
            last_error = exc
            print(f"[load] failed ({type(exc).__name__}): {str(exc)[:220]}")
            gc.collect()
            torch.cuda.empty_cache()

    raise RuntimeError(f"All base-model candidates failed. Last error: {last_error}")


# -----------------------------------------------------------------------------
# 3.3  Pad-token safety (gotcha #1)
# -----------------------------------------------------------------------------
PAD_CANDIDATES = ["<|vision_pad|>", "<|fim_pad|>", "<|image_pad|>", "<|object_ref_start|>"]


def ensure_distinct_pad_token(tokenizer, model) -> None:
    """
    Guarantee pad_token_id != eos_token_id, reusing an existing special token so
    no embedding resize (and therefore no vocab/LoRA mismatch) is needed.
    """
    eos_literal = PROJECT["eos_literal"]
    eos_id = tokenizer.convert_tokens_to_ids(eos_literal)
    if eos_id is None or eos_id == tokenizer.unk_token_id:
        raise RuntimeError(f"{eos_literal} is not in this tokenizer's vocabulary.")

    # Pin EOS to the literal we actually train on. Qwen3's chat EOS is
    # <|im_end|>, which we deliberately do not use (no chat templates).
    tokenizer.eos_token = eos_literal
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.eos_token_id = eos_id
    model.config.eos_token_id = eos_id

    if tokenizer.pad_token_id is not None and tokenizer.pad_token_id != eos_id:
        print(f"[pad] already distinct: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
    else:
        vocab = tokenizer.get_vocab()
        chosen = next((t for t in PAD_CANDIDATES if t in vocab and vocab[t] != eos_id), None)
        if chosen is None:
            tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
            model.resize_token_embeddings(len(tokenizer))
            chosen = "<|pad|>"
            print("[pad] no reusable special token - added <|pad|> and resized embeddings")
        else:
            tokenizer.pad_token = chosen
        print(f"[pad] set pad_token={chosen!r} (id={tokenizer.pad_token_id}) so the real "
              f"EOS at id={eos_id} is NOT masked out of the labels")

    model.config.pad_token_id = tokenizer.pad_token_id
    tokenizer.padding_side = "right"  # right padding for training


def verify_tokenisation(tokenizer, sample_text: str) -> None:
    """Prove that the literal '<|endoftext|>' encodes as ONE special token."""
    ids = tokenizer(sample_text, add_special_tokens=False)["input_ids"]
    eos_id = tokenizer.convert_tokens_to_ids(PROJECT["eos_literal"])
    print(f"\n[tok] sample length      : {len(ids)} tokens")
    print(f"[tok] last 6 token ids   : {ids[-6:]}")
    print(f"[tok] last 6 as strings  : {[tokenizer.decode([i]) for i in ids[-6:]]}")
    assert ids[-1] == eos_id, (
        f"Last token is {ids[-1]} but EOS is {eos_id}. The literal '<|endoftext|>' was split "
        f"into sub-tokens instead of matching the special token - training would teach the "
        f"model to emit that string as text and never actually stop."
    )
    assert ids.count(eos_id) == 1, "EOS appears more than once in a single sample."
    print("[tok] OK - exactly one terminal EOS token per sample")


# -----------------------------------------------------------------------------
# 3.4  Recover datasets if the runtime restarted, then load the base model
# -----------------------------------------------------------------------------
if "train_dataset" not in globals() or "val_dataset" not in globals():
    print("[data] in-memory datasets missing (runtime restart?) - rebuilding from "
          f"{PROJECT['processed_dataset']}")
    from datasets import Dataset as _Dataset

    _rows = [json.loads(line) for line in open(PROJECT["processed_dataset"], encoding="utf-8")]
    # Cell 2 persisted the split label, so the exact same partition is restored
    # rather than re-derived from a shuffle that may not reproduce.
    train_dataset = _Dataset.from_list([r for r in _rows if r.get("split", "train") == "train"])
    val_dataset = _Dataset.from_list([r for r in _rows if r.get("split") == "val"])
    print(f"[data] restored {len(train_dataset)} train / {len(val_dataset)} val")

model, tokenizer, RESOLVED_BASE = load_base_model(PROJECT)
ensure_distinct_pad_token(tokenizer, model)
verify_tokenisation(tokenizer, train_dataset[0]["text"])


# -----------------------------------------------------------------------------
# 3.5  Token-length audit (gotcha #2)
# -----------------------------------------------------------------------------
def audit_token_lengths(tokenizer, dataset, max_seq_length: int,
                        sample_cap: int = 4000) -> int:
    """
    Measure real token lengths and count how many samples would be truncated.
    Truncation removes the terminal EOS, so a nonzero count is a hard problem,
    not a rounding detail.
    """
    texts = dataset["text"][:sample_cap]
    lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in texts]
    lengths.sort()
    n = len(lengths)
    over = sum(1 for x in lengths if x > max_seq_length)
    print(f"\n[len] tokens over {n} samples: p50={lengths[n // 2]}  "
          f"p95={lengths[int(n * 0.95)]}  p99={lengths[int(n * 0.99)]}  max={lengths[-1]}")
    print(f"[len] max_seq_length={max_seq_length} -> {over} sample(s) would be truncated "
          f"({over / n:.2%})")
    if over:
        suggested = 1 << (lengths[-1] - 1).bit_length()      # next power of two
        print(f"[len] WARNING: truncation drops the terminal <|endoftext|> on those samples, "
              f"which teaches the model that stopping is optional.")
        print(f"[len] ACTION : set PROJECT['max_seq_length'] = {suggested} and re-run this cell.")
    else:
        headroom = max_seq_length - lengths[-1]
        print(f"[len] OK - {headroom} tokens of headroom above the longest sample")
    return over


TRUNCATED = audit_token_lengths(tokenizer, train_dataset, PROJECT["max_seq_length"])


# -----------------------------------------------------------------------------
# 3.6  Attach LoRA
# -----------------------------------------------------------------------------
peft_accepted = accepted_params(FastLanguageModel.get_peft_model)
peft_kwargs = {
    "r": PROJECT["lora_r"],
    "lora_alpha": PROJECT["lora_alpha"],
    "lora_dropout": PROJECT["lora_dropout"],       # 0.0 is Unsloth's fast path
    "bias": "none",                                 # "none" is Unsloth's fast path
    "target_modules": PROJECT["target_modules"],    # all 7 linear projections
    "use_gradient_checkpointing": "unsloth",        # ~30% VRAM saving
    "random_state": PROJECT["seed"],
    "use_rslora": False,
    "loftq_config": None,
}
model = FastLanguageModel.get_peft_model(
    model, **{k: v for k, v in peft_kwargs.items() if not peft_accepted or k in peft_accepted}
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n[lora] r={PROJECT['lora_r']} alpha={PROJECT['lora_alpha']} "
      f"-> {trainable:,} trainable / {total:,} total ({trainable / total:.3%})")


# -----------------------------------------------------------------------------
# 3.7  Build the trainer (gotchas #3 and #4)
# -----------------------------------------------------------------------------
def build_trainer(model, tokenizer, train_ds, eval_ds, project: Dict[str, Any]) -> SFTTrainer:
    use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())

    args_cls = SFTConfig if SFTConfig is not None else TrainingArguments
    args_accepted = accepted_params(args_cls.__init__)
    print(f"\n[shim] training-args class : {args_cls.__name__}")

    arg_kwargs: Dict[str, Any] = {
        "output_dir": os.path.join(project["local_root"], "checkpoints"),
        "per_device_train_batch_size": project["micro_batch_size"],
        "gradient_accumulation_steps": project["grad_accum"],
        "num_train_epochs": project["epochs"],
        "learning_rate": project["learning_rate"],
        "lr_scheduler_type": project["lr_scheduler"],
        "warmup_ratio": project["warmup_ratio"],
        "weight_decay": project["weight_decay"],
        "max_grad_norm": project["max_grad_norm"],
        "optim": "adamw_8bit",
        "fp16": not use_bf16,
        "bf16": use_bf16,
        "logging_steps": 10,
        "save_strategy": "epoch",
        "save_total_limit": 3,
        "seed": project["seed"],
        "report_to": "none",
        "gradient_checkpointing": False,   # Unsloth handles this itself
    }

    key = first_supported(args_accepted, "eval_strategy", "evaluation_strategy")
    if key and eval_ds is not None and len(eval_ds):
        arg_kwargs[key] = "epoch"

    key = first_supported(args_accepted, "max_length", "max_seq_length")
    if key:
        arg_kwargs[key] = project["max_seq_length"]

    for name, value in [("dataset_text_field", "text"), ("packing", False), ("dataset_num_proc", 2)]:
        if name in args_accepted:
            arg_kwargs[name] = value

    args = args_cls(**filter_kwargs(args_accepted, arg_kwargs, args_cls.__name__))

    trainer_accepted = accepted_params(SFTTrainer.__init__)
    trainer_kwargs: Dict[str, Any] = {"model": model, "args": args, "train_dataset": train_ds}

    key = first_supported(trainer_accepted, "processing_class", "tokenizer")
    if key:
        trainer_kwargs[key] = tokenizer
    if eval_ds is not None and len(eval_ds) and "eval_dataset" in trainer_accepted:
        trainer_kwargs["eval_dataset"] = eval_ds
    # Older TRL takes these on the trainer instead of the config.
    for name, value in [("dataset_text_field", "text"),
                        ("max_seq_length", project["max_seq_length"]),
                        ("packing", False),
                        ("dataset_num_proc", 2)]:
        if name in trainer_accepted and name not in arg_kwargs:
            trainer_kwargs[name] = value

    trainer = SFTTrainer(**filter_kwargs(trainer_accepted, trainer_kwargs, "SFTTrainer"))

    # Optional: mask the prompt so cross-entropy is computed only on the plan.
    # Usually improves format adherence noticeably, especially at low LR. Off by
    # default to keep the run identical to the specified vanilla-SFT setup.
    if project.get("train_on_responses_only"):
        try:
            from unsloth.chat_templates import train_on_responses_only
            trainer = train_on_responses_only(
                trainer,
                instruction_part="Instruction: Provide a semantic plan without math.\nQuestion:",
                response_part="\nPlan:\n",
            )
            print("[shim] completion-only loss ENABLED (prompt tokens masked)")
        except Exception as exc:
            print(f"[shim] completion-only loss unavailable, using full-sequence loss "
                  f"({type(exc).__name__}: {str(exc)[:120]})")

    return trainer


trainer = build_trainer(model, tokenizer, train_dataset, val_dataset, PROJECT)

steps_per_epoch = max(1, len(train_dataset) // (PROJECT["micro_batch_size"] * PROJECT["grad_accum"]))
print(f"\n[plan] {len(train_dataset)} samples | {steps_per_epoch} optimiser steps/epoch "
      f"| {steps_per_epoch * PROJECT['epochs']} total steps")
if TRUNCATED:
    print(f"[plan] {TRUNCATED} truncated sample(s) present - see the WARNING above before "
          f"trusting the smoke test.")


# -----------------------------------------------------------------------------
# 3.8  Train
# -----------------------------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
start_ts = time.time()

train_result = trainer.train()

elapsed = time.time() - start_ts
peak_gb = torch.cuda.max_memory_reserved() / 1024 ** 3 if torch.cuda.is_available() else 0.0
print(f"\n[train] wall clock : {elapsed / 60:.1f} min")
print(f"[train] peak VRAM  : {peak_gb:.2f} GB")
try:
    print(f"[train] final loss : {train_result.training_loss:.4f}")
except Exception:
    pass


# -----------------------------------------------------------------------------
# 3.9  Save adapter + run manifest, then mirror to Google Drive
# -----------------------------------------------------------------------------
def save_adapter(model, tokenizer, project: Dict[str, Any], base_repo: str,
                 metrics: Dict[str, Any]) -> str:
    out_dir = project["adapter_dir"]
    os.makedirs(out_dir, exist_ok=True)

    model.save_pretrained(out_dir)      # adapter_model.safetensors + adapter_config.json
    tokenizer.save_pretrained(out_dir)  # keeps the corrected pad/eos settings

    # The manifest lets Cell 4 reconstruct this exact setup after a runtime
    # restart without re-deriving anything by hand.
    manifest = {
        "base_model": base_repo,
        "prompt_train": project["prompt_train"],
        "prompt_planner": project["prompt_planner"],
        "prompt_executor": project["prompt_executor"],
        "eos_literal": project["eos_literal"],
        "eos_token_id": tokenizer.convert_tokens_to_ids(project["eos_literal"]),
        "pad_token": tokenizer.pad_token,
        "pad_token_id": tokenizer.pad_token_id,
        "operator_vocab": project.get("operator_vocab", []),
        "max_steps": project.get("max_steps", 9),
        "lora": {k: project[k] for k in ["lora_r", "lora_alpha", "lora_dropout", "target_modules"]},
        "training": {k: project[k] for k in ["learning_rate", "epochs", "lr_scheduler",
                                             "micro_batch_size", "grad_accum", "max_seq_length"]},
        "n_train": len(train_dataset),
        "n_val": len(val_dataset),
        "truncated_samples": int(TRUNCATED),
        "metrics": metrics,
    }
    with open(os.path.join(out_dir, "run_manifest.json"), "w", encoding="utf-8") as fh:
        json.dump(manifest, fh, indent=2)

    size_mb = sum(
        os.path.getsize(os.path.join(out_dir, f))
        for f in os.listdir(out_dir)
        if os.path.isfile(os.path.join(out_dir, f))
    ) / 1024 ** 2
    print(f"\n[save] local  -> {out_dir}  ({size_mb:.1f} MB)")
    return out_dir


def mirror_to_drive(src_dir: str, dst_dir: str) -> None:
    """Mount Drive and copy. Never fatal - a failed backup must not lose the run."""
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
    except Exception as exc:
        print(f"[save] Drive unavailable ({type(exc).__name__}) - local copy retained at {src_dir}")
        return

    try:
        os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
        if os.path.exists(dst_dir):
            backup = f"{dst_dir}_prev"
            shutil.rmtree(backup, ignore_errors=True)
            shutil.move(dst_dir, backup)
            print(f"[save] previous adapter rotated to {backup}")
        shutil.copytree(src_dir, dst_dir)
        print(f"[save] drive  -> {dst_dir}")
    except Exception as exc:
        print(f"[save] Drive copy failed ({type(exc).__name__}: {exc}) - local copy is intact.")


_metrics = {"train_runtime_min": round(elapsed / 60, 2), "peak_vram_gb": round(peak_gb, 2)}
try:
    _metrics["final_train_loss"] = float(train_result.training_loss)
except Exception:
    pass

adapter_path = save_adapter(model, tokenizer, PROJECT, RESOLVED_BASE, _metrics)
mirror_to_drive(adapter_path, PROJECT["drive_adapter_dir"])

PROJECT["resolved_planner_base"] = RESOLVED_BASE
with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
    json.dump(PROJECT, fh, indent=2)


# -----------------------------------------------------------------------------
# 3.10  SMOKE TEST - the acceptance gate before you spend GPU hours on eval
# -----------------------------------------------------------------------------
OPERATOR_VOCAB = set(PROJECT.get("operator_vocab") or
                     ["ADD", "SUBTRACT", "MULTIPLY", "DIVIDE", "CONVERT", "RATIO",
                      "COMPARE", "COUNT"])

# Must match the trained format: "Step 1 [MULTIPLY]: ..."
TAGGED_STEP_RE = re.compile(r"^[ \t]*Step[ \t]*(\d+)[ \t]*\[[ \t]*([A-Z_]+)[ \t]*\][ \t]*:[ \t]*(.+)$",
                            re.MULTILINE)
_ARITH = re.compile(r"\d[ \t]*(?:[+*/×÷])[ \t]*\d|\d[ \t]+-[ \t]+\d|=[ \t]*\$?\d")

SMOKE_QUESTIONS = [
    "A bottle and a cap cost $1.10 in total. The bottle costs $1.00 more than the cap. "
    "What does the cap cost?",
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips "
    "in May. How many clips did Natalia sell altogether in April and May?",
    "A train travels 60 miles in 45 minutes. At the same speed, how many miles will it "
    "travel in 2 hours?",
    "Tom has 3 boxes with 12 pencils each. He gives away 7 pencils and then buys 2 more "
    "boxes. How many pencils does he have?",
    "There are 25 students in a class. 60% play football and 40% play chess, and 5 play "
    "both. How many play neither?",
]


def smoke_test(model, tokenizer, project: Dict[str, Any]) -> None:
    try:
        FastLanguageModel.for_inference(model)  # 2x faster native generation
    except Exception as exc:
        print(f"[smoke] for_inference unavailable ({type(exc).__name__}) - plain generate()")
    model.eval()

    device = next(model.parameters()).device
    eos_id = tokenizer.convert_tokens_to_ids(project["eos_literal"])

    n_format = n_clean = n_ops_valid = 0
    for i, question in enumerate(SMOKE_QUESTIONS, 1):
        prompt = project["prompt_planner"].replace("{question}", question)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=project["planner_max_new_tokens"],
                do_sample=False,
                eos_token_id=eos_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        plan = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()

        steps = TAGGED_STEP_RE.findall(plan)
        fmt_ok = bool(steps) and steps[0][0] == "1"
        clean = _ARITH.search(plan) is None
        ops: List[str] = [op for _, op, _ in steps]
        ops_valid = bool(ops) and all(op in OPERATOR_VOCAB for op in ops)

        n_format += int(fmt_ok)
        n_clean += int(clean)
        n_ops_valid += int(ops_valid)

        print(f"\n[{i}] Q: {question[:76]}...")
        print(f"    tagged_format={fmt_ok}  no_arithmetic={clean}  "
              f"operators_in_vocab={ops_valid}  ops={ops}")
        for line in plan.split("\n")[:8]:
            print(f"    | {line}")

    n = len(SMOKE_QUESTIONS)
    print("\n" + "=" * 76)
    print(f"SMOKE TEST   'Step N [OP]:' format : {n_format}/{n} ({n_format / n:.0%})")
    print(f"             arithmetic-free       : {n_clean}/{n} ({n_clean / n:.0%})")
    print(f"             operators in vocab    : {n_ops_valid}/{n} ({n_ops_valid / n:.0%})")
    if min(n_format, n_clean, n_ops_valid) / n < 0.9:
        print("VERDICT: FAIL. The adapter underfit at lr=2e-5.")
        print("ACTION : set PROJECT['learning_rate']=1e-4 (optionally also")
        print("         PROJECT['train_on_responses_only']=True), re-run Cell 3.")
        print("         Do not run Cell 4 yet - you would be evaluating noise.")
    else:
        print("VERDICT: PASS. Planner emits clean, operator-tagged, calculation-free "
              "guidance. Proceed to Cell 4.")
    print("=" * 76)


smoke_test(model, tokenizer, PROJECT)


[load] attempting unsloth/Qwen3-4B-unsloth-bnb-4bit ...
==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[load] OK -> unsloth/Qwen3-4B-unsloth-bnb-4bit
[pad] already distinct: '<|vision_pad|>' (id=151654)

[tok] sample length      : 161 tokens
[tok] last 6 token ids   : [3368, 594, 1482, 4231, 13, 151643]
[tok] last 6 as strings  : [' mom', "'s", ' current', ' age', '.', '<|endoftext|>']
[tok] OK - exactly one terminal EOS token per sample

[len] tokens over 2850 samples: p50=149  p95=236  p99=276  max=362
[len] max_seq_length=1024 -> 0 sample(s) would be truncated (0.00%)
[len] OK - 662 tokens of headroom above the longest sample


Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



[lora] r=16 alpha=32 -> 33,030,144 trainable / 2,541,616,640 total (1.300%)

[shim] training-args class : SFTConfig


...


[2] Q: Natalia sold clips to 48 of her friends in April, and then she sold half as ...
    tagged_format=False  no_arithmetic=True  operators_in_vocab=False  ops=[]
    | 


Both `max_new_tokens` (=192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[3] Q: A train travels 60 miles in 45 minutes. At the same speed, how many miles wi...
    tagged_format=False  no_arithmetic=True  operators_in_vocab=False  ops=[]
    | Convert the travel time from hours to minutes to match the time unit of the given distance. Determine the distance traveled per minute by dividing the total distance by the total travel time in minutes. Multiply the distance per minute by the total travel time in hours converted to minutes to find the total distance traveled.

[4] Q: Tom has 3 boxes with 12 pencils each. He gives away 7 pencils and then buys ...
    tagged_format=False  no_arithmetic=True  operators_in_vocab=False  ops=[]
    | 


Both `max_new_tokens` (=192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[5] Q: There are 25 students in a class. 60% play football and 40% play chess, and ...
    tagged_format=False  no_arithmetic=True  operators_in_vocab=False  ops=[]
    | 

SMOKE TEST   'Step N [OP]:' format : 0/5 (0%)
             arithmetic-free       : 5/5 (100%)
             operators in vocab    : 0/5 (0%)
VERDICT: FAIL. The adapter underfit at lr=2e-5.
ACTION : set PROJECT['learning_rate']=1e-4 (optionally also
         PROJECT['train_on_responses_only']=True), re-run Cell 3.
         Do not run Cell 4 yet - you would be evaluating noise.


In [ ]:
# =============================================================================
# CELL 3.5 - TOKEN-BOUNDARY DIAGNOSIS & FIX  (run this BEFORE retraining)
# =============================================================================
# Symptom: smoke test format 0/5, several completely empty generations, but the
# prose content is correct semantic guidance with zero arithmetic.
#
# That combination does NOT look like underfitting. Underfitting degrades
# content and format together. Learning the task but not the scaffold points at
# a prompt-boundary problem instead.
#
# Root cause: Qwen's pre-tokenizer regex contains the alternative
#     ?[^\s\p{L}\p{N}]+[\r\n]*
# which greedily absorbs trailing newlines into a punctuation pretoken. So:
#
#     training   "...Plan:\nStep 1 [SUBTRACT]: ..."  -> ['Plan', ':\n', 'Step', ...]
#     inference  "...Plan:"                          -> ['Plan', ':']
#
# ':' and ':\n' are DIFFERENT token ids. The model was trained to emit 'Step'
# immediately after ':\n' and has never once seen ':' in terminal position, so
# at inference it improvises (prose) or takes the highest-probability exit
# (immediate EOS -> your blank outputs).
#
# My Cell 2 assertion checked a CHARACTER-level prefix. Under BPE that is not
# sufficient - the contract that matters is a TOKEN-level prefix. Section 3.5.1
# below is the corrected check and belongs in Cell 2 permanently.
#
# THE FIX IS ONE CHARACTER AND NEEDS NO RETRAINING: append "\n" to the planner
# inference prompt so it tokenises to ['Plan', ':\n'], exactly as in training.
# Your 137 MB adapter is fine. Run this cell against it.
# =============================================================================

import os
import re
import json
import shutil
from typing import Any, Dict, List, Optional, Tuple

import torch

CONFIG_PATH = "/content/sgft_project.json" if os.path.exists("/content/sgft_project.json") else "./sgft_project.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    PROJECT = json.load(fh)


# -----------------------------------------------------------------------------
# 3.5.0  Reuse the in-memory model if Cell 3's session is still alive,
#        otherwise reload the saved adapter. No retraining either way.
# -----------------------------------------------------------------------------
if "model" in globals() and "tokenizer" in globals():
    print("[reuse] using the model/tokenizer already in memory from Cell 3")
else:
    from unsloth import FastLanguageModel
    src = PROJECT["adapter_dir"]
    if not os.path.exists(os.path.join(src, "adapter_config.json")):
        src = PROJECT["drive_adapter_dir"]
    print(f"[reload] loading saved adapter from {src}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=src,
        max_seq_length=PROJECT["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
    )

# Silence the "both max_new_tokens and max_length are set" spam: Qwen3 ships
# max_length=40960 in its generation_config and we always pass max_new_tokens.
if getattr(model, "generation_config", None) is not None:
    model.generation_config.max_length = None

EOS_LITERAL = PROJECT["eos_literal"]
EOS_ID = tokenizer.convert_tokens_to_ids(EOS_LITERAL)


def token_ids(text: str) -> List[int]:
    return tokenizer(text, add_special_tokens=False)["input_ids"]


# -----------------------------------------------------------------------------
# 3.5.1  THE CORRECTED CONTRACT CHECK - token-level, not character-level
# -----------------------------------------------------------------------------
def load_reference_sample(project: Dict[str, Any]) -> Tuple[str, str]:
    """Return (question, full_training_text) for one real training row."""
    if "train_dataset" in globals():
        row = globals()["train_dataset"][0]
        return row["question"], row["text"]
    with open(project["processed_dataset"], "r", encoding="utf-8") as fh:
        row = json.loads(fh.readline())
    return row["question"], row["text"]


def token_prefix_report(project: Dict[str, Any]) -> Optional[str]:
    """
    Test each candidate planner prompt for being a true TOKEN prefix of the
    training string. Returns the winning template, or None if none qualifies.
    """
    question, full_text = load_reference_sample(project)
    full_ids = token_ids(full_text)

    base_template = project["prompt_planner"]
    candidates = {
        "as-specified (ends 'Plan:')":  base_template,
        "newline-terminated ('Plan:\\n')": base_template + "\n",
    }

    print("\n" + "=" * 78)
    print("TOKEN-PREFIX CONTRACT CHECK")
    print("=" * 78)
    print(f"training text tokens : {len(full_ids)}")

    winner = None
    for label, template in candidates.items():
        prompt = template.replace("{question}", question)
        pid = token_ids(prompt)
        is_prefix = full_ids[:len(pid)] == pid
        nxt = full_ids[len(pid)] if len(full_ids) > len(pid) else None

        print(f"\n  {label}")
        print(f"    prompt tokens      : {len(pid)}")
        print(f"    last 4 prompt ids  : {pid[-4:]} -> {[tokenizer.decode([i]) for i in pid[-4:]]!r}")
        print(f"    TRUE TOKEN PREFIX  : {is_prefix}")
        if not is_prefix:
            # Show exactly where the two token streams diverge.
            k = next((i for i in range(min(len(pid), len(full_ids)))
                      if pid[i] != full_ids[i]), min(len(pid), len(full_ids)))
            print(f"    diverges at token  : {k}")
            print(f"      prompt has       : {pid[k:k + 2]} -> "
                  f"{[tokenizer.decode([i]) for i in pid[k:k + 2]]!r}")
            print(f"      training has     : {full_ids[k:k + 2]} -> "
                  f"{[tokenizer.decode([i]) for i in full_ids[k:k + 2]]!r}")
            print(f"    => the model must continue from a token sequence it never saw")
        else:
            print(f"    model must emit next: {nxt} -> {tokenizer.decode([nxt])!r}")
            if winner is None:
                winner = template

    print("\n" + "=" * 78)
    if winner:
        print(f"SELECTED planner prompt: {winner!r}")
    else:
        print("NO candidate is a token prefix. See the guidance printed at the end.")
    print("=" * 78)
    return winner


CORRECT_PLANNER_PROMPT = token_prefix_report(PROJECT)


# -----------------------------------------------------------------------------
# 3.5.2  Patch the config and the saved manifest so Cell 4 picks up the fix
# -----------------------------------------------------------------------------
def patch_planner_prompt(project: Dict[str, Any], new_prompt: str) -> None:
    """Write the corrected prompt to the config and to every saved manifest."""
    project["prompt_planner"] = new_prompt
    project["prompt_planner_token_verified"] = True
    with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
        json.dump(project, fh, indent=2)
    print(f"[patch] {CONFIG_PATH}")

    for adapter_dir in [project["adapter_dir"], project["drive_adapter_dir"]]:
        manifest_path = os.path.join(adapter_dir, "run_manifest.json")
        if not os.path.exists(manifest_path):
            continue
        try:
            with open(manifest_path, "r", encoding="utf-8") as fh:
                manifest = json.load(fh)
            manifest["prompt_planner"] = new_prompt
            manifest["prompt_planner_token_verified"] = True
            with open(manifest_path, "w", encoding="utf-8") as fh:
                json.dump(manifest, fh, indent=2)
            print(f"[patch] {manifest_path}")
        except Exception as exc:
            print(f"[patch] failed on {manifest_path}: {type(exc).__name__}: {exc}")


if CORRECT_PLANNER_PROMPT and CORRECT_PLANNER_PROMPT != PROJECT["prompt_planner"]:
    patch_planner_prompt(PROJECT, CORRECT_PLANNER_PROMPT)
elif CORRECT_PLANNER_PROMPT:
    print("[patch] the as-specified prompt is already a valid token prefix - no change")


# -----------------------------------------------------------------------------
# 3.5.3  Re-run the smoke test with the corrected prompt (no retraining)
# -----------------------------------------------------------------------------
OPERATOR_VOCAB = set(PROJECT.get("operator_vocab") or
                     ["ADD", "SUBTRACT", "MULTIPLY", "DIVIDE", "CONVERT", "RATIO",
                      "COMPARE", "COUNT"])
TAGGED_STEP_RE = re.compile(
    r"^[ \t]*Step[ \t]*(\d+)[ \t]*\[[ \t]*([A-Z_]+)[ \t]*\][ \t]*:[ \t]*(.+)$", re.MULTILINE)
_ARITH = re.compile(r"\d[ \t]*(?:[+*/×÷])[ \t]*\d|\d[ \t]+-[ \t]+\d|=[ \t]*\$?\d")

SMOKE_QUESTIONS = [
    "A bottle and a cap cost $1.10 in total. The bottle costs $1.00 more than the cap. "
    "What does the cap cost?",
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips "
    "in May. How many clips did Natalia sell altogether in April and May?",
    "A train travels 60 miles in 45 minutes. At the same speed, how many miles will it "
    "travel in 2 hours?",
    "Tom has 3 boxes with 12 pencils each. He gives away 7 pencils and then buys 2 more "
    "boxes. How many pencils does he have?",
    "There are 25 students in a class. 60% play football and 40% play chess, and 5 play "
    "both. How many play neither?",
]


def compare_prompts(project: Dict[str, Any], templates: Dict[str, str]) -> Dict[str, Dict[str, int]]:
    """
    Generate with each candidate prompt side by side. Seeing both in one table
    is what turns "the model is broken" into "the prompt was one token short".
    """
    try:
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)
    except Exception:
        pass
    model.eval()
    device = next(model.parameters()).device

    results: Dict[str, Dict[str, int]] = {}
    for label, template in templates.items():
        scores = {"format": 0, "clean": 0, "ops_valid": 0, "empty": 0}
        print("\n" + "#" * 78)
        print(f"# PROMPT VARIANT: {label}   {template.replace('{question}', '<Q>')!r}")
        print("#" * 78)

        for i, question in enumerate(SMOKE_QUESTIONS, 1):
            prompt = template.replace("{question}", question)
            inputs = tokenizer(prompt, return_tensors="pt",
                               add_special_tokens=False).to(device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=project["planner_max_new_tokens"],
                    do_sample=False,
                    temperature=None, top_p=None, top_k=None,
                    eos_token_id=EOS_ID,
                    pad_token_id=tokenizer.pad_token_id,
                )
            plan = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens=True).strip()

            steps = TAGGED_STEP_RE.findall(plan)
            ops = [op for _, op, _ in steps]
            fmt_ok = bool(steps) and steps[0][0] == "1"
            clean = _ARITH.search(plan) is None
            ops_ok = bool(ops) and all(op in OPERATOR_VOCAB for op in ops)

            scores["format"] += int(fmt_ok)
            scores["clean"] += int(clean)
            scores["ops_valid"] += int(ops_ok)
            scores["empty"] += int(not plan)

            print(f"\n[{i}] format={fmt_ok} clean={clean} ops_valid={ops_ok} ops={ops}")
            for line in (plan or "<EMPTY GENERATION>").split("\n")[:8]:
                print(f"    | {line}")

        results[label] = scores

    n = len(SMOKE_QUESTIONS)
    print("\n" + "=" * 78)
    print(f"{'variant':<34} {'format':>8} {'clean':>8} {'ops_ok':>8} {'empty':>8}")
    print("-" * 78)
    for label, s in results.items():
        print(f"{label:<34} {s['format']}/{n:<6} {s['clean']}/{n:<6} "
              f"{s['ops_valid']}/{n:<6} {s['empty']}/{n:<6}")
    print("=" * 78)
    return results


_variants = {"as-specified (ends 'Plan:')": PROJECT.get("prompt_planner_original")
             or "Instruction: Provide a semantic plan without math.\nQuestion: {question}\nPlan:"}
if CORRECT_PLANNER_PROMPT:
    _variants["token-verified"] = CORRECT_PLANNER_PROMPT

SMOKE = compare_prompts(PROJECT, _variants)


# -----------------------------------------------------------------------------
# 3.5.4  Verdict and next action
# -----------------------------------------------------------------------------
n = len(SMOKE_QUESTIONS)
best_label = max(SMOKE, key=lambda k: (SMOKE[k]["format"], SMOKE[k]["ops_valid"]))
best = SMOKE[best_label]

print("\n" + "=" * 78)
if min(best["format"], best["ops_valid"]) / n >= 0.9:
    print(f"VERDICT: PASS with '{best_label}'.")
    print("         The adapter was never the problem - the inference prompt was one")
    print("         token short. No retraining needed.")
    print("ACTION : run Cell 4. It reads prompt_planner from the patched manifest.")
elif best["format"] / n >= 0.5:
    print(f"VERDICT: PARTIAL with '{best_label}' ({best['format']}/{n} format).")
    print("         The boundary fix helped, so the diagnosis holds, but the adapter is")
    print("         also weakly fit at lr=2e-5.")
    print("ACTION : keep the patched prompt, then re-run Cell 3 with")
    print("           PROJECT['learning_rate'] = 1e-4")
    print("           PROJECT['train_on_responses_only'] = True")
    print("         Expect format to go to ~100%.")
else:
    print("VERDICT: FAIL even with a token-verified prompt.")
    print("         The boundary was not the (only) cause - this is genuine underfitting.")
    print("ACTION : re-run Cell 3 with")
    print("           PROJECT['learning_rate'] = 1e-4        # 2e-5 is a full-FT value")
    print("           PROJECT['train_on_responses_only'] = True   # loss on the plan only")
    print("         Both changes are in the config; nothing else needs to move.")
print("=" * 78)


# =============================================================================
# PERMANENT PATCHES - fold these into the notebook so this cannot recur
# =============================================================================
print(r'''
--- PATCH 1: Cell 2, replace the character-level assertion --------------------
The old check was necessary but not sufficient. Add this after it:

    def _ids(t): return tokenizer(t, add_special_tokens=False)["input_ids"]
    _q, _full = train_dataset[0]["question"], train_dataset[0]["text"]
    _p = PROJECT["prompt_planner"].replace("{question}", _q)
    assert _ids(_full)[:len(_ids(_p))] == _ids(_p), (
        "Planner prompt is a character prefix but NOT a token prefix of the "
        "training string. Under BPE those are different things, and only the "
        "token-level one matters."
    )

(Cell 2 has no tokenizer, so either move this check to the top of Cell 3 -
where the tokenizer exists and the dataset is already in scope - or load
AutoTokenizer.from_pretrained(PROJECT["planner_base_candidates"][0]) in Cell 2.
Putting it in Cell 3 right after verify_tokenisation() is simpler.)

--- PATCH 2: Cell 1, prompt_planner ------------------------------------------
    PROMPT_PLANNER = (
        "Instruction: Provide a semantic plan without math.\n"
        "Question: {question}\n"
        "Plan:\n"          # <- trailing newline: makes ':\n' one token, as in training
    )

--- PATCH 3: general rule ----------------------------------------------------
Never end a fine-tuned model's inference prompt on a character that the
tokenizer merges with what follows it. Punctuation followed by whitespace or a
newline is the classic trap: ':', ':\n', '.\n', '"', ' -'. Always assert
token-level prefixing, and always print the last few token ids so a mismatch
is visible rather than inferred.

Phase B is unaffected: it is a frozen base model doing free-form completion, so
there is no training contract for its prompt to violate.
''')

[reuse] using the model/tokenizer already in memory from Cell 3

TOKEN-PREFIX CONTRACT CHECK
training text tokens : 161

  as-specified (ends 'Plan:')
    prompt tokens      : 76
    last 4 prompt ids  : [6981, 5267, 20485, 25] -> [' father', '?\n', 'Plan', ':']
    TRUE TOKEN PREFIX  : False
    diverges at token  : 75
      prompt has       : [25] -> [':']
      training has     : [510, 8304] -> [':\n', 'Step']
    => the model must continue from a token sequence it never saw

  newline-terminated ('Plan:\n')
    prompt tokens      : 76
    last 4 prompt ids  : [6981, 5267, 20485, 510] -> [' father', '?\n', 'Plan', ':\n']
    TRUE TOKEN PREFIX  : True
    model must emit next: 8304 -> 'Step'

SELECTED planner prompt: 'Instruction: Provide a semantic plan without math.\nQuestion: {question}\nPlan:\n'
[patch] /content/sgft_project.json
[patch] /content/sgft/adapters/phaseA_planner/run_manifest.json
[patch] /content/drive/MyDrive/SGFT_2SLM/adapters/phaseA_planner/run_manifest.json

####

## Cell 4 — Pipeline & evaluation engine

Loads Phase A (base + adapter) and Phase B (frozen) with plain `transformers` + `peft` rather
than Unsloth — that avoids Unsloth's global patching interfering when two different models share
VRAM, and lets this cell run standalone after a restart.

Also runs the **un-guided baseline** (executor alone, same answer contract, no plan) in the same
loop, so one pass gives you the guided-vs-direct ablation your report needs — including the
guided-only / baseline-only / both-wrong breakdown.

Selecting a slice:

```python
results = evaluate(500, 600)                        # problems 500–599
results = evaluate(indices=[7, 42, 1300])           # cherry-picked
results = evaluate(0, 1319)                         # full GSM8K test set
results = evaluate(0, 200, run_baseline_too=False)  # guided only, ~2x faster
solve("A bottle and a cap cost $1.10 ...")          # single question, verbose
```

Rows append to `pipeline_detailed_results.csv` with an `fsync` after every problem, so a runtime
timeout never costs you completed work. Re-running with `resume=True` skips finished indices.

In [ ]:
# =============================================================================
# CELL 4 - 2-SLM COLLABORATIVE PIPELINE & EVALUATION ENGINE
# =============================================================================
# Phase A (Planner)  : Qwen3-4B + SGFT LoRA  -> "Step N [OP]: ..." plan, no math
# Phase B (Executor) : Qwen2.5-Math-1.5B     -> frozen base, does the arithmetic
#
# Deliberately independent of Cells 1-3: it re-reads the run manifest from disk
# and uses plain transformers + peft (not Unsloth) to load both models. That
# means it runs cleanly after a runtime restart, and avoids Unsloth's global
# monkey-patching interfering when two different models share VRAM.
#
# The operator tags are the interesting part of this dataset. They give Phase B
# an explicit instruction per step, so this cell tracks three things the plain
# SGFT format cannot: whether every emitted operator is in the closed
# vocabulary, which operators appear, and whether operator validity correlates
# with getting the answer right.
#
# It also runs the un-guided baseline (executor alone) in the same loop, so one
# pass yields the guided-vs-direct ablation the report needs.
# =============================================================================

import os
import re
import csv
import gc
import json
import math
import time
import shutil
import traceback
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence, Tuple

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

# -----------------------------------------------------------------------------
# 4.0  Configuration
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# 4.0  Configuration
# -----------------------------------------------------------------------------
CONFIG_PATH = "/content/sgft_project.json" if os.path.exists("/content/sgft_project.json") else "./sgft_project.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    PROJECT = json.load(fh)

ADAPTER_DIR = PROJECT["adapter_dir"]
if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    ADAPTER_DIR = PROJECT["drive_adapter_dir"]      # fall back to the Drive copy
    print(f"[cfg] local adapter missing, using Drive copy: {ADAPTER_DIR}")

MANIFEST: Dict[str, Any] = {}
MANIFEST_PATH = os.path.join(ADAPTER_DIR, "run_manifest.json")
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r", encoding="utf-8") as fh:
        MANIFEST = json.load(fh)
    print(f"[cfg] manifest loaded from {MANIFEST_PATH}")
else:
    print(f"[cfg] WARNING: no run_manifest.json at {MANIFEST_PATH} - falling back to "
          f"{CONFIG_PATH}, which Cell 1 may have overwritten with pre-fix defaults.")

# Manifest wins over PROJECT: it records what was ACTUALLY trained. PROJECT is
# mutable config that Cell 1 rewrites from scratch on every run, so it can
# regress to defaults (max_steps=6, unfixed prompt_planner, no operator_vocab).
PLANNER_BASE = MANIFEST.get("base_model") or PROJECT.get("resolved_planner_base") \
    or PROJECT["planner_base_candidates"][0]
EXECUTOR_MODEL = PROJECT["executor_model"]
PROMPT_TRAIN = MANIFEST.get("prompt_train") or PROJECT["prompt_train"]
PROMPT_EXECUTOR = PROJECT["prompt_executor"]
EOS_LITERAL = MANIFEST.get("eos_literal", PROJECT["eos_literal"])
MAX_STEPS_KEPT = MANIFEST.get("max_steps", PROJECT.get("max_steps", 9))
OPERATOR_VOCAB = set(MANIFEST.get("operator_vocab") or PROJECT.get("operator_vocab") or
                     ["ADD", "SUBTRACT", "MULTIPLY", "DIVIDE", "CONVERT", "RATIO",
                      "COMPARE", "COUNT"])


# -----------------------------------------------------------------------------
# 4.0.1  Planner-prompt repair - derive it from the training template
# -----------------------------------------------------------------------------
# Do not trust a stored prompt_planner. Derive it from prompt_train, which is
# the thing the weights actually saw, and repair the boundary if needed.
#
# The rule that broke us: a fine-tuned model's prompt must not end where the
# tokenizer would merge the last character with what follows. Qwen's
# pre-tokenizer has the alternative ' ?[^\s\p{L}\p{N}]+[\r\n]*', which absorbs
# trailing newlines into a punctuation token - so "Plan:" tokenises to [':']
# while training saw [':\n']. Different token ids, unseen continuation state.
#
# Tokenizer-free repair: split the training template at the plan boundary and
# move any leading whitespace of the remainder into the prompt. That guarantees
# the prompt ends on a non-whitespace-adjacent boundary regardless of which
# tokenizer is in play. Section 4.1 still proves it at the token level once a
# tokenizer exists.
def resolve_planner_prompt(prompt_train: str, stored: Optional[str]) -> str:
    SENT = "\x00Q\x00"                       # cannot occur in a real question
    train = prompt_train.replace("{input}", SENT)

    if "\nPlan:" not in train:
        raise RuntimeError(
            f"prompt_train has no '\\nPlan:' pivot, so the planner prompt cannot be "
            f"derived from it: {prompt_train!r}"
        )

    head, _, tail = train.partition("\nPlan:")
    prompt = head + "\nPlan:"
    # Absorb whatever whitespace separates the prompt from the first plan token.
    absorbed = tail[:len(tail) - len(tail.lstrip("\r\n \t"))]
    prompt += absorbed
    derived = prompt.replace(SENT, "{question}")

    print(f"[cfg] planner prompt derived from prompt_train")
    print(f"        absorbed separator : {absorbed!r}")
    print(f"        derived            : {derived!r}")
    if stored and stored != derived:
        print(f"        stored (OVERRIDDEN): {stored!r}")
        print(f"        -> the stored prompt would have ended mid-token; using the "
              f"derived one")
    return derived


PROMPT_PLANNER = resolve_planner_prompt(
    PROMPT_TRAIN, MANIFEST.get("prompt_planner") or PROJECT.get("prompt_planner")
)

# Hard invariant: the planner prompt must be a character prefix of the training
# template, and the character following it must not be whitespace. Token-level
# verification happens in 4.1 once a tokenizer is loaded.
_SENT = "\x00Q\x00"
_train_probe = PROMPT_TRAIN.replace("{input}", _SENT)
_plan_probe = PROMPT_PLANNER.replace("{question}", _SENT)
assert _train_probe.startswith(_plan_probe), (
    "Planner prompt is not a prefix of the training template.\n"
    f"  planner : {_plan_probe[-60:]!r}\n"
    f"  training: {_train_probe[:len(_plan_probe)][-60:]!r}"
)
_next_char = _train_probe[len(_plan_probe):len(_plan_probe) + 1]
assert not _next_char.isspace(), (
    f"Planner prompt ends immediately before whitespace ({_next_char!r}). BPE will "
    f"merge that whitespace into the preceding token, so the prompt's final token "
    f"will not match what training saw. Move the whitespace into the prompt."
)

# Ablation-only prompt. Same answer contract as the executor prompt, minus the
# plan, so any accuracy delta is attributable to the guidance and nothing else.
# Phase B is a frozen base model doing free-form completion, so it has no
# training contract for its prompt boundary to violate - ending on ':' is fine.
PROMPT_BASELINE = (
    "Solve the following math problem.\n"
    "Reason step by step, and put your final answer within \\boxed{}.\n\n"
    "Problem: {question}\n\n"
    "Solution:"
)

_stale = MANIFEST.get("prompt_executor")
if _stale and _stale != PROMPT_EXECUTOR:
    print("[cfg] ignoring stale manifest executor prompt (pre-Fix-B):")
    print(f"        manifest: {_stale!r}")
    print(f"        using   : {PROMPT_EXECUTOR!r}")

assert "\\boxed" in PROMPT_EXECUTOR and "\\boxed" in PROMPT_BASELINE, (
    "Executor and baseline prompts must both request \\boxed{}. Qwen2.5-Math "
    "emits #### 0/150 times - asking for it pushes the model out of its trained "
    "distribution and into the self-doubt loop."
)
assert "####" not in PROMPT_EXECUTOR, (
    "Executor prompt still demands the #### contract. Apply Cell 1's \\boxed "
    "version, or re-run Cell 1 so PROJECT['prompt_executor'] is refreshed."
)
assert "{plan}" in PROMPT_EXECUTOR and "{plan}" not in PROMPT_BASELINE, (
    "The plan must appear in the guided prompt and only there."
)

print(f"\n[cfg] planner base   : {PLANNER_BASE}")
print(f"[cfg] planner adapter: {ADAPTER_DIR}")
print(f"[cfg] executor       : {EXECUTOR_MODEL}")
print(f"[cfg] operator vocab : {sorted(OPERATOR_VOCAB)}")
print(f"[cfg] max steps kept : {MAX_STEPS_KEPT}")
print(f"[cfg] planner prompt : {PROMPT_PLANNER!r}")
print(f"[cfg] executor prompt: {PROMPT_EXECUTOR!r}")

# -----------------------------------------------------------------------------
# 4.1  Model loading
# -----------------------------------------------------------------------------

import os
import gc
import json
import sys
from typing import Any, Dict, List, Tuple

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# -----------------------------------------------------------------------------
# 4.1.0  Free whatever Cell 3 left in VRAM
# -----------------------------------------------------------------------------
# The 4B training model plus its optimiser state is still resident if you did
# not restart. Two more models on top of that will OOM a T4.
for _name in ["trainer", "model", "PLANNER", "EXECUTOR"]:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"[vram] after cleanup: {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB allocated")

# -----------------------------------------------------------------------------
# 4.1.1  Detect which backend this process is committed to
# -----------------------------------------------------------------------------
# "unsloth" in sys.modules is the reliable signal: the patch is applied at
# import, so if the module is loaded, transformers is already modified.
UNSLOTH_PATCHED = "unsloth" in sys.modules
FORCE_BACKEND = None          # set to "unsloth" or "transformers" to override

BACKEND = FORCE_BACKEND or ("unsloth" if UNSLOTH_PATCHED else "transformers")
print(f"[backend] unsloth imported in this process : {UNSLOTH_PATCHED}")
print(f"[backend] using                            : {BACKEND}")
if BACKEND == "unsloth":
    print("[backend] both models will load via FastLanguageModel so the patched "
          "attention forward finds apply_qkv.")
    print("[backend] For a clean plain-transformers run instead: Runtime > Restart "
          "session, then run only Cell 4.")


# -----------------------------------------------------------------------------
# 4.1.2  Loaders
# -----------------------------------------------------------------------------
def _bnb_config() -> BitsAndBytesConfig:
    compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) \
        else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )


def load_causal_lm(model_id: str, four_bit: bool = True):
    """Plain-transformers 4-bit load. Only safe when unsloth is NOT imported."""
    kwargs: Dict[str, Any] = {"device_map": "auto", "low_cpu_mem_usage": True}
    try:
        cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
        prequantised = getattr(cfg, "quantization_config", None) is not None
    except Exception:
        prequantised = False

    if four_bit and not prequantised:
        kwargs["quantization_config"] = _bnb_config()
    elif prequantised:
        print(f"  [load] {model_id} is pre-quantised - reusing its embedded config")

    dtype = None
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    # transformers renamed torch_dtype -> dtype; try legacy first, then new.
    for dtype_key in ["torch_dtype", "dtype"]:
        attempt = dict(kwargs)
        if dtype is not None:
            attempt[dtype_key] = dtype
        try:
            model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, **attempt)
            model.eval()
            return model
        except TypeError as exc:
            if "dtype" not in str(exc):
                raise
            print(f"  [load] '{dtype_key}' rejected, retrying with the other spelling")

    raise RuntimeError(f"Could not load {model_id} with either dtype keyword.")


def load_unsloth(model_id: str, max_seq_length: int):
    """
    Unsloth load. Pointing model_name at an adapter directory makes Unsloth read
    adapter_config.json and pull the correct base model automatically, so the
    same call handles Phase A (base + adapter) and Phase B (base only).
    """
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    model.eval()
    return model, tokenizer


def configure_tokenizer(tokenizer, eos_literal: str = EOS_LITERAL):
    """Pin EOS to the literal we trained on and guarantee a usable pad token."""
    eos_id = tokenizer.convert_tokens_to_ids(eos_literal)
    if isinstance(eos_id, int) and eos_id >= 0 and eos_id != tokenizer.unk_token_id:
        tokenizer.eos_token = eos_literal
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"   # correct for generation
    return tokenizer


def stop_token_ids(tokenizer) -> List[int]:
    """All ids that should terminate generation, de-duplicated."""
    ids = set()
    for tok in [EOS_LITERAL, "<|im_end|>", tokenizer.eos_token]:
        if not tok:
            continue
        tid = tokenizer.convert_tokens_to_ids(tok)
        if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id:
            ids.add(tid)
    return sorted(ids) or [tokenizer.eos_token_id]


def _quiet_generation_config(model) -> None:
    """Qwen3 ships max_length=40960; we always pass max_new_tokens. Kill the spam."""
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None


def load_pipeline_models() -> Tuple[Any, Any, Any, Any]:
    max_len = PROJECT["max_seq_length"]

    if BACKEND == "unsloth":
        print(f"\n[load] Phase A (base + adapter) <- {ADAPTER_DIR}")
        planner, planner_tok = load_unsloth(ADAPTER_DIR, max_len)

        print(f"\n[load] Phase B (frozen) <- {EXECUTOR_MODEL}")
        executor, executor_tok = load_unsloth(EXECUTOR_MODEL, max_len)

    else:
        print("\n[load] Phase A base ...")
        planner = load_causal_lm(PLANNER_BASE, four_bit=True)
        print(f"[load] Phase A adapter <- {ADAPTER_DIR}")
        planner = PeftModel.from_pretrained(planner, ADAPTER_DIR, is_trainable=False)
        planner.eval()
        try:
            planner_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
        except Exception:
            planner_tok = AutoTokenizer.from_pretrained(PLANNER_BASE, trust_remote_code=True)

        print(f"\n[load] Phase B (frozen) <- {EXECUTOR_MODEL}")
        executor = load_causal_lm(EXECUTOR_MODEL, four_bit=True)
        executor_tok = AutoTokenizer.from_pretrained(EXECUTOR_MODEL, trust_remote_code=True)

    planner_tok = configure_tokenizer(planner_tok)
    executor_tok = configure_tokenizer(executor_tok)
    _quiet_generation_config(planner)
    _quiet_generation_config(executor)

    # Fail loudly HERE rather than 20 frames deep inside generate(). One tiny
    # forward pass proves the attention path is wired correctly for both models.
    for label, mdl, tok in [("Phase A", planner, planner_tok),
                            ("Phase B", executor, executor_tok)]:
        try:
            with torch.no_grad():
                probe = tok("Question: test\n", return_tensors="pt",
                            add_special_tokens=False).to(next(mdl.parameters()).device)
                mdl(**probe)
            print(f"[check] {label} forward pass OK")
        except AttributeError as exc:
            other = "transformers" if BACKEND == "unsloth" else "unsloth"
            raise RuntimeError(
                f"{label} failed its forward-pass probe: {exc}\n\n"
                f"This is the unsloth/transformers patch mismatch. Fix:\n"
                f"  Runtime > Restart session, then run ONLY Cell 4 "
                f"(it is standalone and reads config from disk).\n"
                f"Alternatively set FORCE_BACKEND = {other!r} above and re-run."
            ) from exc

    if torch.cuda.is_available():
        print(f"\n[vram] allocated {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB / "
              f"reserved {torch.cuda.memory_reserved() / 1024 ** 3:.2f} GB")
    return planner, planner_tok, executor, executor_tok



# -----------------------------------------------------------------------------
# 4.2  Generation
# -----------------------------------------------------------------------------
def truncate_at(text: str, markers: Sequence[str]) -> str:
    """Cut the string at the earliest marker. Base models love to keep going."""
    cut = len(text)
    for marker in markers:
        pos = text.find(marker)
        if pos != -1:
            cut = min(cut, pos)
    return text[:cut].strip()


@torch.no_grad()
def generate(model, tokenizer, prompt: str, max_new_tokens: int,
             stop_markers: Sequence[str] = (), repetition_penalty: float = 1.0) -> str:
    """Greedy, deterministic completion of `prompt`. Returns new text only."""
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    prompt_len = inputs["input_ids"].shape[1]

    kwargs: Dict[str, Any] = {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "temperature": None,       # explicit None silences the greedy-decode warning
        "top_p": None,
        "top_k": None,
        "repetition_penalty": repetition_penalty,
        "eos_token_id": stop_token_ids(tokenizer),
        "pad_token_id": tokenizer.pad_token_id,
    }
    if stop_markers:
        kwargs["stop_strings"] = list(stop_markers)
        kwargs["tokenizer"] = tokenizer

    try:
        out = model.generate(**inputs, **kwargs)
    except TypeError:
        # transformers < 4.39 has no stop_strings; fall back to post-hoc cutting.
        kwargs.pop("stop_strings", None)
        kwargs.pop("tokenizer", None)
        out = model.generate(**inputs, **kwargs)

    text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    return truncate_at(text, stop_markers) if stop_markers else text.strip()


# -----------------------------------------------------------------------------
# 4.3  Plan post-processing for the "Step N [OP]: desc" format
# -----------------------------------------------------------------------------
THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
# Operator tag optional at parse time: the model may drop it, and we want to
# measure how often that happens rather than throw the plan away.
STEP_RE = re.compile(
    r"^[ \t]*Step[ \t]*(\d+)[ \t]*(?:\[[ \t]*([^\]]{1,32}?)[ \t]*\])?[ \t]*:[ \t]*(.*?)[ \t]*$",
    re.MULTILINE | re.IGNORECASE,
)
STEP_SPLIT_RE = re.compile(r"[ \t]*(Step[ \t]*\d+[ \t]*(?:\[|:))", re.IGNORECASE)
ARITH_RE = re.compile(
    r"\d[ \t]*(?:[+*/×÷]|\*\*)[ \t]*\d"
    r"|\d[ \t]+-[ \t]+\d"
    r"|\d[ \t]*[xX][ \t]*\d"
    r"|=[ \t]*\$?\d"
    r"|<<[^>]*>>"
)
PLAN_STOPS = ("\nQuestion:", "\nInstruction:", "Instruction:", EOS_LITERAL, "<|im_end|>")


def sanitise_plan(raw: str) -> Dict[str, Any]:
    """
    Turn whatever the planner emitted into a clean, renumbered, operator-tagged
    plan and report diagnostics. Qwen3 can emit <think> blocks even without a
    chat template, so those are stripped first.
    """
    text = THINK_RE.sub("", raw or "")
    text = re.sub(r"^\s*<think>", "", text, flags=re.IGNORECASE)   # unterminated block
    text = truncate_at(text, PLAN_STOPS)
    text = re.sub(r"^\s*Plan\s*:\s*", "", text, flags=re.IGNORECASE)
    text = STEP_SPLIT_RE.sub(r"\n\1", text)

    steps: List[Tuple[str, str]] = []          # (operator, description)
    for _num, op, desc in STEP_RE.findall(text):
        desc = " ".join(desc.split())
        if desc:
            steps.append(((op or "").strip().upper(), desc))
    steps = steps[:MAX_STEPS_KEPT]

    degraded = not steps
    if degraded:
        # The planner produced prose. Keep it so Phase B still gets context,
        # but flag it - plan_format_ok=False in the CSV.
        lines = [ln.strip() for ln in text.split("\n") if ln.strip()][:MAX_STEPS_KEPT]
        plan = "\n".join(lines)
        ops: List[str] = []
    else:
        plan = "\n".join(f"Step {i} [{op}]: {desc}" if op else f"Step {i}: {desc}"
                         for i, (op, desc) in enumerate(steps, 1))
        ops = [op for op, _ in steps]

    tagged = [op for op in ops if op]
    invalid = sorted({op for op in tagged if op not in OPERATOR_VOCAB})

    return {
        "plan": plan,
        "n_steps": len(steps) if not degraded else 0,
        "format_ok": (not degraded) and len(tagged) == len(ops) and len(ops) > 0,
        "operators": ops,
        "invalid_operators": invalid,
        "ops_all_valid": bool(tagged) and not invalid and len(tagged) == len(ops),
        "untagged_steps": len(ops) - len(tagged),
        "has_math": bool(ARITH_RE.search(plan)),
        "empty": not plan,
    }


# -----------------------------------------------------------------------------
# 4.4  Answer extraction - the part that quietly decides your reported accuracy
# -----------------------------------------------------------------------------
NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\.\d+")
CALC_ANNOT_RE = re.compile(r"<<[^>]*>>")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")
HASH_RE = re.compile(r"####\s*([^\n]*)")
PHRASE_RE = re.compile(
    r"(?:final\s+answer|answer|result|total|therefore|so\s+the\s+answer)"
    # Connectors may chain ("answer is:", "answer =", "result equals"), so this
    # repeats rather than matching a single optional token.
    r"(?:\s*(?:\b(?:is|are|was|were|be|equals)\b|[:=]))*"
    r"\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
    re.IGNORECASE,
)
EQUALS_RESULT_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")
ANSWER_PHRASE_RE = re.compile(
    r"(?:final\s+answer|answer|result|therefore|so\s+the\s+answer)"
    r"(?:\s*(?:\b(?:is|are|was|were|be|equals)\b|[:=]))*"
    r"\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
    re.IGNORECASE,
)
RAMBLE_MARKERS = (
    "however", "wait", "but ", "but,", "let's correct", "let me correct",
    "there seems to be", "there is a mistake", "actually,", "on second thought",
    "hold on", "recalculat", "re-calculat", "let's re", "let me re",
)

def _ramble_cut(text: str) -> int:
    """Index of the first self-doubt marker, or len(text) if none."""
    low = text.lower()
    cut = len(text)
    for marker in RAMBLE_MARKERS:
        pos = low.find(marker)
        if pos != -1:
            cut = min(cut, pos)
    return cut

def _modal_number(text: str):
    """Most frequent number, ties broken by earliest appearance."""
    from collections import Counter
    vals = []
    for m in NUM_RE.finditer(text):
        v = to_float(m.group(0))
        if v is not None:
            vals.append(v)
    if not vals:
        return None
    counts = Counter(vals)
    best_val, best_ct = None, 1
    for v in vals:
        if counts[v] > best_ct:
            best_val, best_ct = v, counts[v]
    return (best_val, best_ct) if best_val is not None else None

def to_float(token):
    """'1,234.50' / '$42' / '7.' -> float, else None."""
    if token is None:
        return None
    cleaned = str(token).strip().replace(",", "").replace("$", "").replace("%", "")
    cleaned = cleaned.replace("\\", "").strip().rstrip(".").strip()
    if not cleaned:
        return None
    try:
        value = float(cleaned)
    except ValueError:
        return None
    return value if math.isfinite(value) else None


def _first_number(fragment: str):
    match = NUM_RE.search(fragment or "")
    return to_float(match.group(0)) if match else None


def extract_answer(text):
    """
    Ramble-robust extraction. Returns (value, strategy).

    Order matters and was determined empirically on your own failing rows:
      1. ####                    the contract, when honoured
      2. \\boxed{}               what Qwen-Math emits natively
      3. strict answer phrase    pre-ramble only
      4. last '= result'         pre-ramble; beats modal because operands
                                 repeat MORE than results in worked arithmetic
      5. modal number            prose fallback, no equations
      6-10. whole-text variants, then the old last-number behaviour
    """
    if not text:
        return None, "empty"
    cleaned = CALC_ANNOT_RE.sub("", text)

    hashes = HASH_RE.findall(cleaned)
    if hashes:
        v = _first_number(hashes[-1])
        if v is not None:
            return v, "hash"

    boxed = BOXED_RE.findall(cleaned)
    if boxed:
        v = _first_number(boxed[-1])
        if v is not None:
            return v, "boxed"

    pre = cleaned[:_ramble_cut(cleaned)]

    phrases = ANSWER_PHRASE_RE.findall(pre)
    if phrases:
        v = to_float(phrases[-1])
        if v is not None:
            return v, "phrase_preramble"

    eq = EQUALS_RESULT_RE.findall(pre)
    if eq:
        v = to_float(eq[-1])
        if v is not None:
            return v, "equals_preramble"

    modal = _modal_number(pre)
    if modal and modal[1] >= 2:
        return modal[0], "modal_preramble"

    phrases = ANSWER_PHRASE_RE.findall(cleaned)
    if phrases:
        v = to_float(phrases[-1])
        if v is not None:
            return v, "phrase"

    modal = _modal_number(cleaned)
    if modal and modal[1] >= 2:
        return modal[0], "modal"

    eq = EQUALS_RESULT_RE.findall(cleaned)
    if eq:
        v = to_float(eq[-1])
        if v is not None:
            return v, "equals"

    nums_pre = NUM_RE.findall(pre)
    if nums_pre:
        v = to_float(nums_pre[-1])
        if v is not None:
            return v, "last_number_preramble"

    nums = NUM_RE.findall(cleaned)
    if nums:
        v = to_float(nums[-1])
        if v is not None:
            return v, "last_number"

    return None, "none"


def extract_gold(answer_field: str) -> Optional[float]:
    """GSM8K gold answers always end with '#### <number>'."""
    tail = answer_field.split("####")[-1] if "####" in answer_field else answer_field
    return _first_number(CALC_ANNOT_RE.sub("", tail))


def answers_match(pred: Optional[float], gold: Optional[float]) -> bool:
    """Float-tolerant comparison: 0.05 == 0.0500, 18 == 18.0."""
    if pred is None or gold is None:
        return False
    return math.isclose(pred, gold, rel_tol=1e-4, abs_tol=1e-4)


# -----------------------------------------------------------------------------
# 4.5  GSM8K loading with flexible indexing
# -----------------------------------------------------------------------------
def load_gsm8k_test():
    for repo in ["openai/gsm8k", "gsm8k"]:
        try:
            ds = load_dataset(repo, "main", split="test")
            print(f"[data] GSM8K test loaded from '{repo}' ({len(ds)} problems)")
            return ds
        except Exception as exc:
            print(f"[data] '{repo}' failed: {type(exc).__name__}")
    raise RuntimeError("Could not load the GSM8K test split. Check network access.")


def select_indices(dataset, start: int = 0, end: Optional[int] = None,
                   indices: Optional[Sequence[int]] = None,
                   limit: Optional[int] = None) -> List[int]:
    """
    Flexible selection:
        select_indices(ds, 500, 600)                -> 500..599
        select_indices(ds, indices=[3, 17, 902])    -> exactly those
        select_indices(ds, limit=50)                -> first 50
    """
    n = len(dataset)
    if indices is not None:
        chosen = [i for i in indices if 0 <= i < n]
        skipped = len(indices) - len(chosen)
        if skipped:
            print(f"[data] {skipped} out-of-range indices ignored (valid range 0..{n - 1})")
    else:
        end = n if end is None else min(end, n)
        start = max(0, start)
        chosen = list(range(start, end))
    if limit is not None:
        chosen = chosen[:limit]
    return chosen


# -----------------------------------------------------------------------------
# 4.6  CSV logging (incremental, crash-safe, resumable)
# -----------------------------------------------------------------------------
CSV_FIELDS = [
    "idx", "question", "gold",
    "plan", "plan_steps", "plan_format_ok", "plan_operators",
    "plan_invalid_operators", "plan_ops_all_valid", "plan_untagged_steps", "plan_has_math",
    "guided_raw", "guided_pred", "guided_strategy", "guided_correct",
    "baseline_raw", "baseline_pred", "baseline_strategy", "baseline_correct",
    "t_planner_s", "t_executor_s", "t_baseline_s", "error",
]


def init_csv(path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        with open(path, "w", newline="", encoding="utf-8") as fh:
            csv.DictWriter(fh, fieldnames=CSV_FIELDS).writeheader()


def append_csv(path: str, row: Dict[str, Any]) -> None:
    """Append + fsync after every problem so a crash never loses completed work."""
    with open(path, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
        writer.writerow({k: row.get(k, "") for k in CSV_FIELDS})
        fh.flush()
        os.fsync(fh.fileno())


def completed_indices(path: str) -> set:
    if not os.path.exists(path):
        return set()
    done = set()
    try:
        with open(path, "r", newline="", encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                if row.get("idx", "").strip().isdigit() and not row.get("error"):
                    done.add(int(row["idx"]))
    except Exception as exc:
        print(f"[csv] could not read existing results ({type(exc).__name__}) - starting fresh")
    return done


# -----------------------------------------------------------------------------
# 4.7  The two-stage pipeline
# -----------------------------------------------------------------------------
EXEC_STOPS = ("\nProblem:", "\nQuestion:", "\nInstruction:", "\nPlan:",
              "\nSolution:", "<|im_end|>")
FALLBACK_PLAN = ("Step 1 [COUNT]: Identify the quantities given in the problem.\n"
                 "Step 2 [COMPARE]: Determine the relationship between them.\n"
                 "Step 3 [ADD]: Compute the requested quantity.")


def run_planner(question: str) -> Tuple[Dict[str, Any], str, float]:
    prompt = PROMPT_PLANNER.replace("{question}", question)
    t0 = time.time()
    raw = generate(PLANNER, PLANNER_TOK, prompt,
                   max_new_tokens=PROJECT["planner_max_new_tokens"],
                   stop_markers=PLAN_STOPS)
    return sanitise_plan(raw), raw, time.time() - t0


def run_executor(question: str, plan: str) -> Tuple[str, float]:
    # A blank plan leaves a dangling "Plan:\n\n" in the prompt, which base
    # models handle badly. Substitute a neutral scaffold; plan_format_ok in the
    # CSV records that Phase A failed here.
    plan = plan.strip() or FALLBACK_PLAN
    prompt = PROMPT_EXECUTOR.replace("{question}", question).replace("{plan}", plan)
    t0 = time.time()
    text = generate(EXECUTOR, EXECUTOR_TOK, prompt,
                    max_new_tokens=PROJECT["executor_max_new_tokens"],
                    stop_markers=EXEC_STOPS,
                    repetition_penalty=PROJECT["repetition_penalty"])
    return text, time.time() - t0


def run_baseline(question: str) -> Tuple[str, float]:
    prompt = PROMPT_BASELINE.replace("{question}", question)
    t0 = time.time()
    text = generate(EXECUTOR, EXECUTOR_TOK, prompt,
                    max_new_tokens=PROJECT["executor_max_new_tokens"],
                    stop_markers=EXEC_STOPS,
                    repetition_penalty=PROJECT["repetition_penalty"])
    return text, time.time() - t0


def solve(question: str, verbose: bool = True) -> Dict[str, Any]:
    """Single-question convenience wrapper - handy for demos and the viva."""
    plan_info, _raw, t_a = run_planner(question)
    answer_text, t_b = run_executor(question, plan_info["plan"])
    pred, strategy = extract_answer(answer_text)
    if verbose:
        print(f"\nQ: {question}\n\n--- PHASE A plan  (ops={plan_info['operators']}) ---\n"
              f"{plan_info['plan']}\n\n--- PHASE B answer ---\n{answer_text}\n\n"
              f"Parsed: {pred} (via {strategy})   [{t_a:.1f}s + {t_b:.1f}s]")
    return {"plan": plan_info["plan"], "operators": plan_info["operators"],
            "answer_text": answer_text, "prediction": pred,
            "strategy": strategy, "latency_s": t_a + t_b}


# -----------------------------------------------------------------------------
# 4.8  Evaluation loop
# -----------------------------------------------------------------------------
def _short(text: str, width: int = 100) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[:width - 3] + "..."


def evaluate(start: int = 0, end: Optional[int] = 100,
             indices: Optional[Sequence[int]] = None,
             limit: Optional[int] = None,
             run_baseline_too: bool = True,
             resume: bool = True,
             csv_path: Optional[str] = None,
             verbose: bool = True) -> Dict[str, Any]:
    """
    Evaluate the two-model pipeline on a slice of the GSM8K test set.

        evaluate(500, 600)                     # problems 500-599
        evaluate(indices=[7, 42, 1300])        # cherry-picked
        evaluate(limit=25, run_baseline_too=False)
    """
    csv_path = csv_path or PROJECT["results_csv"]
    init_csv(csv_path)

    targets = select_indices(GSM8K, start=start, end=end, indices=indices, limit=limit)
    already = completed_indices(csv_path) if resume else set()
    pending = [i for i in targets if i not in already]

    print("\n" + "=" * 78)
    print(f"EVALUATION  requested={len(targets)}  already_done={len(targets) - len(pending)}  "
          f"to_run={len(pending)}")
    print(f"CSV -> {csv_path}")
    print("=" * 78)

    stats: Dict[str, Any] = {
        "n": 0, "guided_correct": 0, "baseline_correct": 0, "errors": 0,
        "format_ok": 0, "ops_all_valid": 0, "plan_has_math": 0, "untagged_steps": 0,
        "both_correct": 0, "guided_only": 0, "baseline_only": 0, "both_wrong": 0,
        "t_planner": 0.0, "t_executor": 0.0, "t_baseline": 0.0,
        "strategies": Counter(), "operators": Counter(), "invalid_operators": Counter(),
        "correct_by_steps": Counter(), "total_by_steps": Counter(),
    }
    t_start = time.time()

    try:
        for position, idx in enumerate(pending, 1):
            record = GSM8K[idx]
            question, gold_field = record["question"], record["answer"]
            gold = extract_gold(gold_field)

            row: Dict[str, Any] = {"idx": idx, "question": question, "gold": gold, "error": ""}

            try:
                plan_info, _raw_plan, t_a = run_planner(question)
                answer_text, t_b = run_executor(question, plan_info["plan"])
                pred, strategy = extract_answer(answer_text)
                guided_ok = answers_match(pred, gold)

                base_text, base_pred, base_strategy, base_ok, t_c = "", None, "", False, 0.0
                if run_baseline_too:
                    base_text, t_c = run_baseline(question)
                    base_pred, base_strategy = extract_answer(base_text)
                    base_ok = answers_match(base_pred, gold)

                row.update({
                    "plan": plan_info["plan"],
                    "plan_steps": plan_info["n_steps"],
                    "plan_format_ok": plan_info["format_ok"],
                    "plan_operators": " ".join(plan_info["operators"]),
                    "plan_invalid_operators": " ".join(plan_info["invalid_operators"]),
                    "plan_ops_all_valid": plan_info["ops_all_valid"],
                    "plan_untagged_steps": plan_info["untagged_steps"],
                    "plan_has_math": plan_info["has_math"],
                    "guided_raw": answer_text,
                    "guided_pred": pred,
                    "guided_strategy": strategy,
                    "guided_correct": guided_ok,
                    "baseline_raw": base_text,
                    "baseline_pred": base_pred,
                    "baseline_strategy": base_strategy,
                    "baseline_correct": base_ok,
                    "t_planner_s": round(t_a, 2),
                    "t_executor_s": round(t_b, 2),
                    "t_baseline_s": round(t_c, 2),
                })

                stats["n"] += 1
                stats["guided_correct"] += int(guided_ok)
                stats["baseline_correct"] += int(base_ok)
                stats["format_ok"] += int(plan_info["format_ok"])
                stats["ops_all_valid"] += int(plan_info["ops_all_valid"])
                stats["plan_has_math"] += int(plan_info["has_math"])
                stats["untagged_steps"] += plan_info["untagged_steps"]
                stats["t_planner"] += t_a
                stats["t_executor"] += t_b
                stats["t_baseline"] += t_c
                stats["strategies"][strategy] += 1
                stats["operators"].update(plan_info["operators"])
                stats["invalid_operators"].update(plan_info["invalid_operators"])
                stats["total_by_steps"][plan_info["n_steps"]] += 1
                if guided_ok:
                    stats["correct_by_steps"][plan_info["n_steps"]] += 1
                if run_baseline_too:
                    key = ("both_correct" if guided_ok and base_ok else
                           "guided_only" if guided_ok else
                           "baseline_only" if base_ok else "both_wrong")
                    stats[key] += 1

                if verbose:
                    acc = stats["guided_correct"] / stats["n"]
                    mark = "PASS" if guided_ok else "FAIL"
                    print("\n" + "-" * 78)
                    print(f"[{position}/{len(pending)}]  test_idx={idx}   {mark}   "
                          f"cumulative accuracy = {stats['guided_correct']}/{stats['n']} "
                          f"({acc:.1%})")
                    print(f"  Q        : {_short(question, 150)}")
                    print(f"  PLAN (A) : steps={plan_info['n_steps']} "
                          f"format_ok={plan_info['format_ok']} "
                          f"ops_valid={plan_info['ops_all_valid']} "
                          f"math_leak={plan_info['has_math']} "
                          f"ops={plan_info['operators']}")
                    for line in plan_info["plan"].split("\n"):
                        print(f"             | {_short(line, 120)}")
                    print(f"  EXEC (B) : {_short(answer_text, 220)}")
                    print(f"  RESULT   : expected={gold}  got={pred}  (via {strategy})"
                          + (f"   | baseline={base_pred} {'PASS' if base_ok else 'FAIL'}"
                             if run_baseline_too else ""))
                    print(f"  TIMING   : A={t_a:.1f}s  B={t_b:.1f}s"
                          + (f"  base={t_c:.1f}s" if run_baseline_too else ""))

            except Exception as exc:
                stats["errors"] += 1
                row["error"] = f"{type(exc).__name__}: {exc}"
                print(f"\n[ERROR] idx={idx} -> {row['error']}")
                traceback.print_exc(limit=2)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            append_csv(csv_path, row)

    except KeyboardInterrupt:
        print("\n\n[interrupt] stopping early - all completed rows are already on disk.")

    stats["elapsed_min"] = (time.time() - t_start) / 60
    summarise(stats, run_baseline_too, csv_path)
    return stats


def summarise(stats: Dict[str, Any], with_baseline: bool, csv_path: str) -> None:
    n = stats["n"]
    print("\n" + "=" * 78)
    print("SUMMARY")
    print("=" * 78)
    if n == 0:
        print("  No problems were scored.")
        return

    guided_acc = stats["guided_correct"] / n
    print(f"  problems scored        : {n}   (errors: {stats['errors']})")
    print(f"  GUIDED accuracy        : {stats['guided_correct']}/{n} = {guided_acc:.2%}")
    if with_baseline:
        base_acc = stats["baseline_correct"] / n
        print(f"  BASELINE accuracy      : {stats['baseline_correct']}/{n} = {base_acc:.2%}")
        print(f"  DELTA (guidance gain)  : {guided_acc - base_acc:+.2%}")
        print(f"    both correct         : {stats['both_correct']}")
        print(f"    guided only (gain)   : {stats['guided_only']}")
        print(f"    baseline only (loss) : {stats['baseline_only']}")
        print(f"    both wrong           : {stats['both_wrong']}")

    print("\n  Phase A plan quality")
    print(f"    'Step N [OP]:' format: {stats['format_ok']}/{n} = {stats['format_ok'] / n:.1%}")
    print(f"    operators all in vocab: {stats['ops_all_valid']}/{n} "
          f"= {stats['ops_all_valid'] / n:.1%}")
    print(f"    untagged steps total : {stats['untagged_steps']}")
    print(f"    arithmetic leakage   : {stats['plan_has_math']}/{n} "
          f"= {stats['plan_has_math'] / n:.1%}   (want ~0%: the planner must not compute)")

    if stats["operators"]:
        total_ops = sum(stats["operators"].values())
        print("\n  Operator distribution in generated plans")
        for op, count in stats["operators"].most_common():
            tag = "" if op in OPERATOR_VOCAB else "  <- OUT OF VOCAB"
            print(f"    {op:<12}: {count:>5}  ({count / total_ops:.1%}){tag}")

    if stats["total_by_steps"]:
        print("\n  Accuracy by plan length (does more decomposition help?)")
        for k in sorted(stats["total_by_steps"]):
            tot = stats["total_by_steps"][k]
            cor = stats["correct_by_steps"].get(k, 0)
            print(f"    {k} steps      : {cor:>4}/{tot:<4} = {cor / tot:.1%}")

    print("\n  Answer-parse strategy mix (a large 'last_number' share means the "
          "'#### N' contract is being ignored)")
    for strategy, count in stats["strategies"].most_common():
        print(f"    {strategy:<14}: {count:>5}  ({count / n:.1%})")

    print("\n  Latency (mean per problem)")
    print(f"    Phase A              : {stats['t_planner'] / n:.2f}s")
    print(f"    Phase B              : {stats['t_executor'] / n:.2f}s")
    print(f"    two-stage total      : {(stats['t_planner'] + stats['t_executor']) / n:.2f}s")
    if with_baseline and stats["t_baseline"] > 0:
        single = stats["t_baseline"] / n
        two = (stats["t_planner"] + stats["t_executor"]) / n
        print(f"    single-stage base    : {single:.2f}s   -> pipeline overhead {two / single:.2f}x")
    print(f"    wall clock           : {stats['elapsed_min']:.1f} min")

    print(f"\n  results CSV            : {csv_path}")

    # Best-effort mirror to Drive so a runtime timeout cannot lose the numbers.
    try:
        dst = PROJECT["drive_results_csv"]
        if os.path.ismount("/content/drive"):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(csv_path, dst)
            print(f"  mirrored to Drive      : {dst}")
    except Exception as exc:
        print(f"  [warn] Drive mirror skipped: {type(exc).__name__}")
    print("=" * 78)


# -----------------------------------------------------------------------------
# 4.9  Boot + run
# -----------------------------------------------------------------------------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PLANNER, PLANNER_TOK, EXECUTOR, EXECUTOR_TOK = load_pipeline_models()
GSM8K = load_gsm8k_test()

# Sanity check on one known-hard problem before committing to a long run.
print("\n" + "#" * 78)
print("# SINGLE-PROBLEM SANITY CHECK")
print("#" * 78)
solve("A bottle and a cap cost $1.10 in total. The bottle costs $1.00 more than "
      "the cap. What does the cap cost?")

# ---- Main evaluation ---------------------------------------------------------
# Change the slice here. Examples:
import pandas as pd
prev = pd.read_csv("/content/sgft/results/pipeline_detailed_results.csv")
results = evaluate(indices=sorted(prev["idx"].dropna().astype(int).tolist()),
                    resume=False,
                    csv_path="/content/sgft/results/run2_boxed.csv")                      # problems 500-599
#   results = evaluate(indices=[0, 5, 99, 1200])       # specific problems
#   results = evaluate(0, 1319)                        # the full GSM8K test set
#   results = evaluate(0, 200, run_baseline_too=False) # guided only, 2x faster
# results = evaluate(start=0, end=50, run_baseline_too=True, resume=True, verbose=True)

[cfg] manifest loaded from /content/sgft/adapters/phaseA_planner/run_manifest.json
[cfg] planner prompt derived from prompt_train
        absorbed separator : '\n'
        derived            : 'Instruction: Provide a semantic plan without math.\nQuestion: {question}\nPlan:\n'
[cfg] ignoring stale manifest executor prompt (pre-Fix-B):
        manifest: 'Question: {question}\nPlan:\n{plan}\nFollow the plan exactly. End your response with #### [number].\nAnswer:'
        using   : 'Solve the following math problem. A step-by-step plan is provided; follow it.\nReason step by step, and put your final answer within \\boxed{}.\n\nProblem: {question}\n\nPlan:\n{plan}\n\nSolution:'

[cfg] planner base   : unsloth/Qwen3-4B-unsloth-bnb-4bit
[cfg] planner adapter: /content/sgft/adapters/phaseA_planner
[cfg] executor       : Qwen/Qwen2.5-Math-1.5B
[cfg] operator vocab : ['ADD', 'COMPARE', 'CONVERT', 'COUNT', 'DIVIDE', 'MULTIPLY', 'RATIO', 'SUBTRACT']
[cfg] max steps kept : 9
[cfg] planner prompt : 

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.



[load] Phase B (frozen) <- Qwen/Qwen2.5-Math-1.5B
==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

[data] GSM8K test loaded from 'openai/gsm8k' (1319 problems)

##############################################################################
# SINGLE-PROBLEM SANITY CHECK
##############################################################################

Q: A bottle and a cap cost $1.10 in total. The bottle costs $1.00 more than the cap. What does the cap cost?

--- PHASE A plan  (ops=['SUBTRACT', 'DIVIDE']) ---
Step 1 [SUBTRACT]: Determine the cost of the cap by subtracting the difference between the bottle and the cap from the total cost.
Step 2 [DIVIDE]: Determine the cost of the cap by dividing the remaining amount by the number of caps.

--- PHASE B answer ---
Let's solve this step by step:

Step 1 [SUBTRACT]:
The bottle costs $1.00 more than the cap. So, we can represent the cost of the cap as x dollars. Therefore, the cost of the bottle would be (x + $1.00) dollars.

Now, let's set up an equation to find the value of x:
Total cost = Cost of bottle + Cost of cap
$1.10 = (x + $1.00) +

In [ ]:
# =============================================================================
# FIX A - RAMBLE-ROBUST ANSWER EXTRACTION + OFFLINE RE-SCORE
# =============================================================================
# Two parts:
#   Part 1  drop-in replacement for extract_answer() (paste over section 4.4's
#           version in Cell 4; keep NUM_RE / to_float / _first_number etc.)
#   Part 2  a standalone cell that re-scores your EXISTING CSV with the new
#           extractor - no GPU, no re-running the models. guided_raw and
#           baseline_raw are already saved, so this recovers your true accuracy
#           in seconds.
#
# Why the old 'last_number' rule was actively wrong: a model that rambles states
# the correct answer FIRST, then degenerates ("However, there seems to be a
# mistake...") and trails off on a stray number. Taking the last number is the
# worst possible choice for exactly the outputs that need help. The new order:
#
#   1. #### N                     (the contract; unambiguous when present)
#   2. \boxed{N}                  (what Qwen-Math emits when it complies)
#   3. explicit answer phrase     (FIRST occurrence, not last - pre-ramble)
#   4. modal final-sentence number(the number the model repeats most)
#   5. last number                (unchanged last-ditch fallback, now labelled)
#
# On the Jill example ($57,500 stated twice, no #### / no \boxed, then a stray
# "3"), step 4 recovers 57500 by frequency where step 5 grabbed the 3.
# =============================================================================

import re
import math
from collections import Counter
from typing import List, Optional, Tuple

# --- shared primitives (identical to Cell 4; included so this cell is runnable
#     on its own for Part 2) -------------------------------------------------
NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\.\d+")
CALC_ANNOT_RE = re.compile(r"<<[^>]*>>")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")
HASH_RE = re.compile(r"####\s*([^\n]*)")

# "= $57,500" / "= 450" at the tail of a calculation line. The RESULT of an
# equation is far more reliable than a fuzzy keyword match, which tends to grab
# the first operand after the keyword rather than the answer.
EQUALS_RESULT_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")

# Loose keyword phrase, used only as a LATE fallback. It can grab operands
# ("salary is $700 + ...") so it must not outrank modal / equals-result.
ANSWER_PHRASE_RE = re.compile(
    r"(?:final\s+answer|answer|result|therefore|so\s+the\s+answer)"
    r"(?:\s*(?:\b(?:is|are|was|were|be|equals)\b|[:=]))*"
    r"\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
    re.IGNORECASE,
)

# Text that marks the start of a model second-guessing itself. Everything from
# here on is degeneration and must not override an answer stated before it.
RAMBLE_MARKERS = (
    "however", "wait", "but ", "but,", "let's correct", "let me correct",
    "there seems to be", "there is a mistake", "actually,", "on second thought",
    "hold on", "recalculat", "re-calculat", "let's re", "let me re",
)


def to_float(token: str) -> Optional[float]:
    """'1,234.50' / '$42' / '7.' -> float, else None."""
    if token is None:
        return None
    cleaned = str(token).strip().replace(",", "").replace("$", "").replace("%", "")
    cleaned = cleaned.replace("\\", "").strip().rstrip(".").strip()
    if not cleaned:
        return None
    try:
        value = float(cleaned)
    except ValueError:
        return None
    return value if math.isfinite(value) else None


def _first_number(fragment: str) -> Optional[float]:
    match = NUM_RE.search(fragment or "")
    return to_float(match.group(0)) if match else None


def _ramble_cut(text: str) -> int:
    """Index of the first self-doubt marker, or len(text) if none. Case-insensitive."""
    low = text.lower()
    cut = len(text)
    for marker in RAMBLE_MARKERS:
        pos = low.find(marker)
        if pos != -1:
            cut = min(cut, pos)
    return cut


def _modal_number(text: str) -> Optional[Tuple[float, int]]:
    """
    The most frequently stated number, tie-broken by earliest appearance.
    A model that reaches the right answer and then loops tends to REPEAT that
    answer, so the mode is a strong signal on degenerate output. Trivial values
    (0, 1, small counts that appear once) are ignored by the count>=2 gate.
    """
    vals: List[float] = []
    for m in NUM_RE.finditer(text):
        v = to_float(m.group(0))
        if v is not None:
            vals.append(v)
    if not vals:
        return None
    counts = Counter(vals)
    best_val, best_ct = None, 1
    for v in vals:                      # iterate in appearance order for tie-break
        c = counts[v]
        if c > best_ct:
            best_val, best_ct = v, c
    return (best_val, best_ct) if best_val is not None else None


def extract_answer(text: str) -> Tuple[Optional[float], str]:
    """
    Ramble-robust extraction. Returns (value, strategy) so the CSV records HOW
    each answer was found. Strategy names are stable across the pipeline.
    """
    if not text:
        return None, "empty"
    cleaned = CALC_ANNOT_RE.sub("", text)

    # 1. The explicit contract. Prefer the LAST #### (final answer line).
    hashes = HASH_RE.findall(cleaned)
    if hashes:
        v = _first_number(hashes[-1])
        if v is not None:
            return v, "hash"

    # 2. \boxed{} - what Qwen-Math emits when it complies. Last boxed wins.
    boxed = BOXED_RE.findall(cleaned)
    if boxed:
        v = _first_number(boxed[-1])
        if v is not None:
            return v, "boxed"

    # 3. Explicit "final answer" phrase BEFORE any self-doubt. Strict keyword
    #    set only (answer/result/therefore), so it won't fire on "salary is".
    pre = cleaned[:_ramble_cut(cleaned)]
    phrases = ANSWER_PHRASE_RE.findall(pre)
    if phrases:
        v = to_float(phrases[-1])
        if v is not None:
            return v, "phrase_preramble"

    # 4. Result of the LAST equation before the ramble ("... = $57,500"). In a
    #    worked solution the final line's '= result' is the answer. This beats
    #    modal because operands ($20, 35, $700) repeat MORE than the result -
    #    the Jill case has 700/450/1150 twice each but 57,500 (the answer) once.
    eq = EQUALS_RESULT_RE.findall(pre)
    if eq:
        v = to_float(eq[-1])
        if v is not None:
            return v, "equals_preramble"

    # 5. Modal number in the pre-ramble region - fallback when there are no
    #    equations (prose answers). Ties break to earliest appearance.
    modal = _modal_number(pre)
    if modal and modal[1] >= 2:
        return modal[0], "modal_preramble"

    # 6. Same phrase search over the whole text if nothing fired pre-ramble.
    phrases = ANSWER_PHRASE_RE.findall(cleaned)
    if phrases:
        v = to_float(phrases[-1])
        if v is not None:
            return v, "phrase"

    # 7. Modal over the whole text.
    modal = _modal_number(cleaned)
    if modal and modal[1] >= 2:
        return modal[0], "modal"

    # 8. Last equation result anywhere.
    eq = EQUALS_RESULT_RE.findall(cleaned)
    if eq:
        v = to_float(eq[-1])
        if v is not None:
            return v, "equals"

    # 9. Last-ditch: last number in the PRE-RAMBLE region (not the whole text,
    #    so trailing degeneration can't hijack it). Labelled distinctly.
    nums_pre = NUM_RE.findall(pre)
    if nums_pre:
        v = to_float(nums_pre[-1])
        if v is not None:
            return v, "last_number_preramble"

    # 10. Absolute fallback: last number anywhere (the old behaviour). A large
    #     share here still means outputs need the format fix (B).
    nums = NUM_RE.findall(cleaned)
    if nums:
        v = to_float(nums[-1])
        if v is not None:
            return v, "last_number"

    return None, "none"


# =============================================================================
# PART 2 - RE-SCORE THE EXISTING CSV (no GPU)
# =============================================================================
def rescore_csv(csv_path: Optional[str] = None,
                gsm8k=None,
                write_back: bool = True) -> dict:
    """
    Recompute guided/baseline correctness from saved *_raw columns using the new
    extractor. Prints old-vs-new accuracy and the strategy shift, and (by
    default) writes a *_rescored.csv alongside the original.

    gsm8k: optional GSM8K test split for gold re-derivation. If None, uses the
    'gold' column already in the CSV.
    """
    import os
    import csv as _csv

    if csv_path is None:
        csv_path = PROJECT["results_csv"] if "PROJECT" in globals() \
            else "/content/sgft/results/pipeline_detailed_results.csv"

    with open(csv_path, "r", newline="", encoding="utf-8") as fh:
        rows = list(_csv.DictReader(fh))
    if not rows:
        print(f"[rescore] {csv_path} is empty")
        return {}

    def _gold_for(row):
        if gsm8k is not None and row.get("idx", "").isdigit():
            ans = gsm8k[int(row["idx"])]["answer"]
            tail = ans.split("####")[-1]
            return _first_number(CALC_ANNOT_RE.sub("", tail))
        return to_float(row.get("gold", ""))

    def _match(pred, gold):
        if pred is None or gold is None:
            return False
        return math.isclose(pred, gold, rel_tol=1e-4, abs_tol=1e-4)

    scored = [r for r in rows if not r.get("error")]
    n = len(scored)

    old_g = sum(str(r.get("guided_correct", "")).strip().lower() == "true" for r in scored)
    old_b = sum(str(r.get("baseline_correct", "")).strip().lower() == "true" for r in scored)

    new_g = new_b = 0
    recovered, lost = [], []
    strat_old, strat_new = Counter(), Counter()
    has_baseline = any(r.get("baseline_raw", "").strip() for r in scored)

    for r in scored:
        gold = _gold_for(r)
        strat_old[r.get("guided_strategy", "?")] += 1

        g_pred, g_strat = extract_answer(r.get("guided_raw", ""))
        g_ok = _match(g_pred, gold)
        strat_new[g_strat] += 1
        new_g += int(g_ok)

        r["guided_pred_rescored"] = g_pred
        r["guided_strategy_rescored"] = g_strat
        r["guided_correct_rescored"] = g_ok

        was = str(r.get("guided_correct", "")).strip().lower() == "true"
        if g_ok and not was:
            recovered.append(r)
        elif was and not g_ok:
            lost.append(r)

        if has_baseline:
            b_pred, b_strat = extract_answer(r.get("baseline_raw", ""))
            b_ok = _match(b_pred, gold)
            new_b += int(b_ok)
            r["baseline_pred_rescored"] = b_pred
            r["baseline_strategy_rescored"] = b_strat
            r["baseline_correct_rescored"] = b_ok

    print("=" * 74)
    print(f"RE-SCORE  {csv_path}")
    print(f"  problems scored : {n}   (skipped {len(rows) - n} error rows)")
    print("-" * 74)
    print(f"  GUIDED   old {old_g}/{n} = {old_g / n:.1%}   ->   "
          f"new {new_g}/{n} = {new_g / n:.1%}   ({new_g - old_g:+d})")
    if has_baseline:
        print(f"  BASELINE old {old_b}/{n} = {old_b / n:.1%}   ->   "
              f"new {new_b}/{n} = {new_b / n:.1%}   ({new_b - old_b:+d})")
        print(f"  DELTA    old {(old_g - old_b) / n:+.1%}   ->   new {(new_g - new_b) / n:+.1%}")
    print("-" * 74)
    print(f"  recovered (was wrong, now right): {len(recovered)}")
    print(f"  lost      (was right, now wrong): {len(lost)}   <- should be ~0")

    print("\n  guided strategy shift (old -> new):")
    for s in sorted(set(strat_old) | set(strat_new)):
        print(f"    {s:<24} {strat_old.get(s, 0):>4} -> {strat_new.get(s, 0):<4}")

    if recovered:
        print("\n  sample recovered rows (parser was throwing these away):")
        for r in recovered[:5]:
            raw = " ".join(r.get("guided_raw", "").split())
            print(f"    idx={r.get('idx'):<5} gold={r.get('gold')} "
                  f"now={r['guided_pred_rescored']} via {r['guided_strategy_rescored']}")
            print(f"        raw: {raw[:150]}...")
    if lost:
        print("\n  [!] LOST rows (new extractor regressed these - inspect):")
        for r in lost[:5]:
            print(f"    idx={r.get('idx')} gold={r.get('gold')} "
                  f"old_pred={r.get('guided_pred')} new_pred={r['guided_pred_rescored']} "
                  f"via {r['guided_strategy_rescored']}")

    if write_back:
        out = csv_path.replace(".csv", "_rescored.csv")
        extra = ["guided_pred_rescored", "guided_strategy_rescored", "guided_correct_rescored"]
        if has_baseline:
            extra += ["baseline_pred_rescored", "baseline_strategy_rescored",
                      "baseline_correct_rescored"]
        fields = list(rows[0].keys()) + [c for c in extra if c not in rows[0]]
        with open(out, "w", newline="", encoding="utf-8") as fh:
            w = _csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
            w.writeheader()
            w.writerows(rows)
        print(f"\n  wrote -> {out}")
    print("=" * 74)

    return {"n": n, "old_guided": old_g, "new_guided": new_g,
            "old_baseline": old_b, "new_baseline": new_b,
            "recovered": len(recovered), "lost": len(lost)}


# -----------------------------------------------------------------------------
# Run it. Finds the CSV wherever it survived (local disk or Drive).
# -----------------------------------------------------------------------------
def _find_results_csv() -> Optional[str]:
    import os
    candidates = []
    if "PROJECT" in globals():
        candidates += [PROJECT.get("results_csv"), PROJECT.get("drive_results_csv")]
    candidates += [
        "/content/sgft/results/pipeline_detailed_results.csv",
        "/content/drive/MyDrive/SGFT_2SLM/results/pipeline_detailed_results.csv",
        "./pipeline_detailed_results.csv",
    ]
    for path in candidates:
        if path and os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[rescore] using {path}")
            return path
    print("[rescore] no results CSV found. Checked:")
    for path in candidates:
        if path:
            print(f"    {path}")
    print("[rescore] If the Colab VM was recycled, run CELL 0 (session recovery) "
          "first to copy it back from Drive.")
    return None


_csv_path = _find_results_csv()
if _csv_path:
    # GSM8K may already be in memory from Cell 4; pass it to re-derive gold
    # from the dataset rather than trusting the CSV's gold column.
    _gsm = GSM8K if "GSM8K" in globals() else None
    rescore_summary = rescore_csv(csv_path=_csv_path, gsm8k=_gsm)
else:
    rescore_summary = {}

[rescore] using /content/sgft/results/pipeline_detailed_results.csv
RE-SCORE  /content/sgft/results/pipeline_detailed_results.csv
  problems scored : 150   (skipped 0 error rows)
--------------------------------------------------------------------------
  GUIDED   old 54/150 = 36.0%   ->   new 78/150 = 52.0%   (+24)
  BASELINE old 51/150 = 34.0%   ->   new 73/150 = 48.7%   (+22)
  DELTA    old +2.0%   ->   new +3.3%
--------------------------------------------------------------------------
  recovered (was wrong, now right): 31
  lost      (was right, now wrong): 7   <- should be ~0

  guided strategy shift (old -> new):
    boxed                      35 -> 35  
    equals_preramble            0 -> 91  
    hash                        2 -> 2   
    last_number                99 -> 0   
    last_number_preramble       0 -> 14  
    modal                       0 -> 1   
    modal_preramble             0 -> 2   
    none                        4 -> 4   
    phrase                     10 -

In [ ]:
# =============================================================================
# FIX A.1 - INSPECT THE LOST ROWS AND TUNE THE RAMBLE CUT
# =============================================================================
# Result from the re-score: 31 recovered, 7 lost. Net +24 is strongly positive,
# but 7 regressions means the ramble cut is firing too early on some rows and
# `equals_preramble` is then grabbing an intermediate instead of the answer.
#
# Prime suspect: RAMBLE_MARKERS contains "but " and "but,". Those are ordinary
# words in mathematical prose ("But we still need to subtract the discount"),
# not self-correction signals. If the cut fires mid-reasoning, everything after
# the real final calculation is discarded.
#
# This cell does two things, both on data you already have - no GPU:
#   PART 1  prints the full raw output of every lost row so you can see the
#           actual failure, not a hypothesis about it
#   PART 2  ablates several marker sets against all 150 rows and reports which
#           maximises accuracy, so the choice is measured rather than argued
#
# Run it, read the table, then set RAMBLE_MARKERS in Cell 4 sec 4.4 to the
# winner.
# =============================================================================

import os
import re
import csv
import math
from collections import Counter
from typing import Dict, List, Optional, Tuple

# -----------------------------------------------------------------------------
# A.1.0  Locate the re-scored CSV
# -----------------------------------------------------------------------------
def _find_csv() -> str:
    candidates = []
    if "PROJECT" in globals():
        candidates += [PROJECT.get("results_csv"), PROJECT.get("drive_results_csv")]
    candidates += [
        "/content/sgft/results/pipeline_detailed_results.csv",
        "/content/drive/MyDrive/SGFT_2SLM/results/pipeline_detailed_results.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p) and os.path.getsize(p) > 0:
            print(f"[tune] using {p}")
            return p
    raise FileNotFoundError("No results CSV found.")


CSV_PATH = _find_csv()
with open(CSV_PATH, "r", newline="", encoding="utf-8") as fh:
    ROWS = [r for r in csv.DictReader(fh) if not r.get("error")]
print(f"[tune] {len(ROWS)} scored rows")


# -----------------------------------------------------------------------------
# A.1.1  Parametrised extractor - same logic, swappable marker set
# -----------------------------------------------------------------------------
NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\.\d+")
CALC_ANNOT_RE = re.compile(r"<<[^>]*>>")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")
HASH_RE = re.compile(r"####\s*([^\n]*)")
EQUALS_RESULT_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")
ANSWER_PHRASE_RE = re.compile(
    r"(?:final\s+answer|answer|result|therefore|so\s+the\s+answer)"
    r"(?:\s*(?:\b(?:is|are|was|were|be|equals)\b|[:=]))*"
    r"\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)",
    re.IGNORECASE,
)

# --- marker set variants -----------------------------------------------------
MARKERS_CURRENT = (
    "however", "wait", "but ", "but,", "let's correct", "let me correct",
    "there seems to be", "there is a mistake", "actually,", "on second thought",
    "hold on", "recalculat", "re-calculat", "let's re", "let me re",
)

# Drops bare "but" (ordinary prose) and "actually," (often a legitimate aside).
MARKERS_NARROW = (
    "however", "wait,", "let's correct", "let me correct",
    "there seems to be", "there is a mistake", "on second thought",
    "hold on", "recalculat", "re-calculat", "let's re", "let me re",
)

# Only unambiguous self-correction. "however" alone is common in valid prose
# ("However, the discount applies only to..."), so it must co-occur with a
# mistake cue to count.
MARKERS_STRICT = (
    "there seems to be a mistake", "there is a mistake", "there was a mistake",
    "let's correct", "let me correct", "correct it:", "recalculat",
    "re-calculat", "on second thought", "hold on", "i made an error",
    "that's incorrect", "this is incorrect",
)

MARKERS_NONE: Tuple[str, ...] = ()

VARIANTS = {
    "current":  MARKERS_CURRENT,
    "narrow":   MARKERS_NARROW,
    "strict":   MARKERS_STRICT,
    "no_cut":   MARKERS_NONE,
}


def to_float(token) -> Optional[float]:
    if token is None:
        return None
    c = str(token).strip().replace(",", "").replace("$", "").replace("%", "")
    c = c.replace("\\", "").strip().rstrip(".").strip()
    if not c:
        return None
    try:
        v = float(c)
    except ValueError:
        return None
    return v if math.isfinite(v) else None


def _first_number(frag: str) -> Optional[float]:
    m = NUM_RE.search(frag or "")
    return to_float(m.group(0)) if m else None


def _cut(text: str, markers: Tuple[str, ...]) -> int:
    low = text.lower()
    cut = len(text)
    for mk in markers:
        p = low.find(mk)
        if p != -1:
            cut = min(cut, p)
    return cut


def _modal(text: str):
    vals = [to_float(m.group(0)) for m in NUM_RE.finditer(text)]
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    counts = Counter(vals)
    best, bc = None, 1
    for v in vals:
        if counts[v] > bc:
            best, bc = v, counts[v]
    return (best, bc) if best is not None else None


def extract(text: str, markers: Tuple[str, ...],
            last_line_bias: bool = False) -> Tuple[Optional[float], str]:
    """Same ladder as Fix A, with a swappable marker set.

    last_line_bias: before falling back to the last equation anywhere in the
    pre-ramble region, try the last equation on the FINAL non-empty line. Models
    that restate the answer at the end benefit; those that trail off do not.
    """
    if not text:
        return None, "empty"
    cleaned = CALC_ANNOT_RE.sub("", text)

    h = HASH_RE.findall(cleaned)
    if h:
        v = _first_number(h[-1])
        if v is not None:
            return v, "hash"

    b = BOXED_RE.findall(cleaned)
    if b:
        v = _first_number(b[-1])
        if v is not None:
            return v, "boxed"

    pre = cleaned[:_cut(cleaned, markers)]

    p = ANSWER_PHRASE_RE.findall(pre)
    if p:
        v = to_float(p[-1])
        if v is not None:
            return v, "phrase_preramble"

    if last_line_bias:
        lines = [ln for ln in pre.split("\n") if ln.strip()]
        if lines:
            eq_last = EQUALS_RESULT_RE.findall(lines[-1])
            if eq_last:
                v = to_float(eq_last[-1])
                if v is not None:
                    return v, "equals_lastline"

    eq = EQUALS_RESULT_RE.findall(pre)
    if eq:
        v = to_float(eq[-1])
        if v is not None:
            return v, "equals_preramble"

    m = _modal(pre)
    if m and m[1] >= 2:
        return m[0], "modal_preramble"

    p = ANSWER_PHRASE_RE.findall(cleaned)
    if p:
        v = to_float(p[-1])
        if v is not None:
            return v, "phrase"

    m = _modal(cleaned)
    if m and m[1] >= 2:
        return m[0], "modal"

    eq = EQUALS_RESULT_RE.findall(cleaned)
    if eq:
        v = to_float(eq[-1])
        if v is not None:
            return v, "equals"

    n = NUM_RE.findall(pre)
    if n:
        v = to_float(n[-1])
        if v is not None:
            return v, "last_number_preramble"

    n = NUM_RE.findall(cleaned)
    if n:
        v = to_float(n[-1])
        if v is not None:
            return v, "last_number"

    return None, "none"


def _match(a, b):
    return a is not None and b is not None and math.isclose(a, b, rel_tol=1e-4, abs_tol=1e-4)


# -----------------------------------------------------------------------------
# PART 1 - show the lost rows in full
# -----------------------------------------------------------------------------
LOST_IDX = {"24", "937", "951", "958", "988"}   # from your re-score output

print("\n" + "=" * 78)
print("PART 1 - LOST ROWS (new extractor regressed these)")
print("=" * 78)
for r in ROWS:
    if r.get("idx") not in LOST_IDX:
        continue
    gold = to_float(r.get("gold"))
    raw = r.get("guided_raw", "")
    cur_v, cur_s = extract(raw, MARKERS_CURRENT)
    nar_v, _ = extract(raw, MARKERS_NARROW)
    str_v, _ = extract(raw, MARKERS_STRICT)
    non_v, _ = extract(raw, MARKERS_NONE)

    cut_at = _cut(raw, MARKERS_CURRENT)
    trigger = "none"
    low = raw.lower()
    for mk in MARKERS_CURRENT:
        if low.find(mk) == cut_at:
            trigger = mk
            break

    print(f"\n--- idx={r['idx']}  gold={gold} " + "-" * 45)
    print(f"  current={cur_v} ({cur_s})   narrow={nar_v}   strict={str_v}   no_cut={non_v}")
    print(f"  ramble cut at char {cut_at}/{len(raw)} triggered by {trigger!r}")
    print(f"  --- text kept (pre-ramble) ---")
    print("   " + raw[:cut_at].strip().replace("\n", "\n   ")[:900])
    print(f"  --- text DISCARDED ---")
    disc = raw[cut_at:].strip()
    print("   " + (disc.replace("\n", "\n   ")[:600] if disc else "<nothing>"))


# -----------------------------------------------------------------------------
# PART 2 - ablate marker sets over all rows
# -----------------------------------------------------------------------------
print("\n\n" + "=" * 78)
print("PART 2 - MARKER-SET ABLATION over all scored rows")
print("=" * 78)

configs = []
for name, mk in VARIANTS.items():
    configs.append((name, mk, False))
    configs.append((name + "+lastline", mk, True))

results = {}
for name, mk, bias in configs:
    g_ok = b_ok = 0
    strat = Counter()
    for r in ROWS:
        gold = to_float(r.get("gold"))
        v, s = extract(r.get("guided_raw", ""), mk, bias)
        strat[s] += 1
        g_ok += int(_match(v, gold))
        if r.get("baseline_raw", "").strip():
            bv, _ = extract(r["baseline_raw"], mk, bias)
            b_ok += int(_match(bv, gold))
    results[name] = (g_ok, b_ok, strat)

n = len(ROWS)
print(f"\n{'variant':<24} {'guided':>14} {'baseline':>14} {'delta':>9}")
print("-" * 78)
best_name, best_score = None, -1
for name, (g, b, _) in results.items():
    print(f"{name:<24} {g:>4}/{n} = {g / n:>5.1%} {b:>4}/{n} = {b / n:>5.1%} "
          f"{(g - b) / n:>+8.1%}")
    if g > best_score:
        best_name, best_score = name, g

print("-" * 78)
print(f"BEST guided accuracy: {best_name}  ({best_score}/{n} = {best_score / n:.1%})")
print("\nstrategy mix for the winner:")
for s, c in results[best_name][2].most_common():
    print(f"   {s:<24} {c:>4}  ({c / n:.1%})")

print(f"""
--- WHAT TO DO --------------------------------------------------------------
Set RAMBLE_MARKERS in Cell 4 section 4.4 to the '{best_name.split('+')[0]}' set above.
{"Also enable last_line_bias (add the equals_lastline branch)." if "lastline" in best_name else ""}

Caveat worth stating in your report: this tunes a parser on the same 150 rows
you are reporting accuracy for, so the winning number is mildly optimistic.
The clean version is to fix the OUTPUT FORMAT (Fix B) so `boxed`/`hash` carry
most of the mass and the heuristics rarely fire at all. Right now
equals_preramble carries 61% - that is the real problem, and no amount of
marker tuning removes it.
-----------------------------------------------------------------------------""")

[tune] using /content/sgft/results/pipeline_detailed_results.csv
[tune] 150 scored rows

PART 1 - LOST ROWS (new extractor regressed these)

--- idx=24  gold=26.0 ---------------------------------------------
  current=14.625 (equals_preramble)   narrow=14.625   strict=26.0   no_cut=26.0
  ramble cut at char 470/1095 triggered by 'however'
  --- text kept (pre-ramble) ---
   The original price of the book was $26.00.
   
   Explanation:
   Step 1 [MULTIPLY]: To find the discount amount, we multiply the discounted price by the discount rate. In this case, the discounted price is $19.50 and the discount rate is 25%, or 0.25 in decimal form. So, the discount amount is $19.50 * 0.25 = $4.875.
   Step 2 [SUBTRACT]: To find the original price, we subtract the discount amount from the discounted price. So, the original price is $19.50 - $4.875 = $14.625.
  --- text DISCARDED ---
   However, since prices are typically rounded to the nearest cent, the original price would be $14.63. But, consid

In [ ]:
# =============================================================================
# CELL 5 (FIXED) - GUIDANCE INTERNALIZATION, 2-SLM -> 1-SLM
# =============================================================================
# The previous run was invalid. Three compounding bugs, all fixed here:
#
#   BUG 1  M1 loss was nan for all 3 epochs (fp16 overflow on T4; the log said
#          Bfloat16 = FALSE). The adapter learned nothing.
#          FIX: force fp32 training. Slower but numerically stable on T4, and at
#          1.5B/QLoRA it still fits. Also skip-on-nan guard as a backstop.
#
#   BUG 2  "Removed 7098 of 7100 samples where all labels were -100". TRL's
#          completion-only masker could not find the response marker, masked
#          every token, and the filter then DELETED the rows. M2/M3 trained on
#          2 samples in 0.1 min - untrained.
#          FIX: do the masking MANUALLY in a pre-tokenized dataset. We know
#          exactly where the answer starts because we build the prompt and the
#          answer separately, so there is no string-matching to fail. Loss on
#          the answer tokens only, guaranteed, with an assert that proves it.
#
#   BUG 3  completion-only was reported "unavailable (ValueError)" for M1, so it
#          silently used full-sequence loss - a different objective from the
#          (broken) M2/M3. FIX: same manual masking path for all three, so the
#          objective is identical.
#
# Plus an eval bug: the parser took the FIRST number. The models were emitting
# correct answers inside chain-of-thought ("16 - 3 - 4 = 9\n9 x 2 = 18") and
# being scored on "16". FIX below uses the ramble-robust extractor, and the
# whole point of training is to make the model terse so this barely matters.
#
# This version does NOT use trl.SFTTrainer or train_on_responses_only at all.
# It uses a plain Trainer with a pre-masked dataset, which removes every moving
# part that failed last time.
# =============================================================================

from unsloth import FastLanguageModel  # before transformers/torch

import os
import re
import gc
import json
import time
import math
import shutil
from typing import Any, Dict, List, Optional, Tuple

import torch
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

CONFIG_PATH = "/content/sgft_project.json" if os.path.exists("/content/sgft_project.json") else "./sgft_project.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    PROJECT = json.load(fh)

EOS_LITERAL = PROJECT["eos_literal"]


# =============================================================================
# 5.0  CONFIG
# =============================================================================
NT = {
    "base_candidates": ["unsloth/Qwen2.5-Math-1.5B-bnb-4bit",
                        "unsloth/Qwen2.5-Math-1.5B",
                        "Qwen/Qwen2.5-Math-1.5B"],
    "learning_rate": 1e-4,
    "lr_stage2": 5e-5,
    "epochs": 3,
    "lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "max_seq_length": 640,
    "micro_batch_size": 2, "grad_accum": 8,
    "seed": PROJECT.get("seed", 3407), "val_ratio": 0.05,
    "data_path": "/content/sgft/data/nothink_dataset.jsonl",
    "out_root": "/content/sgft/adapters",
    "drive_root": PROJECT.get("drive_root", "/content/drive/MyDrive/SGFT_2SLM"),
    # fp32 on T4: bf16 unsupported, fp16 gave nan. ~1.4x slower, numerically safe.
    "force_fp32": True,
}

# Prompts split at the answer boundary so masking is exact, not string-matched.
# {prefix} is everything the model conditions on; {answer} is all it is scored on.
SCAFFOLD_PREFIX = ("Solve the problem using the plan.\n"
                   "Question: {question}\n"
                   "Plan:\n{plan}\n"
                   "Answer:\n")
DIRECT_PREFIX = ("Solve the problem.\n"
                 "Question: {question}\n"
                 "Answer:\n")
# training target appended to the prefix:  {answer} + EOS


# =============================================================================
# 5.1  BUILD DATASET (no GPU) - unchanged, it worked (7473 joined, 0 unmatched)
# =============================================================================
def _norm_q(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def _fmt_answer(v: float) -> str:
    return str(int(v)) if v == int(v) else f"{v:g}"


def build_nothink_dataset(force: bool = False) -> List[Dict[str, str]]:
    if os.path.exists(NT["data_path"]) and not force:
        rows = [json.loads(l) for l in open(NT["data_path"], encoding="utf-8")]
        print(f"[data] cached {len(rows)} rows")
        return rows
    sg_path = next((p for p in PROJECT["dataset_search_paths"] + ["/content/sg_dataset_improv.json"]
                    if p and os.path.exists(p)), None)
    if sg_path is None:
        raise FileNotFoundError("sg_dataset_improv.json not found")
    sg = json.load(open(sg_path, encoding="utf-8"))
    train = load_dataset("openai/gsm8k", "main", split="train")
    gold = {}
    for r in train:
        m = re.search(r"[-+]?\d[\d,]*(?:\.\d+)?", r["answer"].split("####")[-1])
        if m:
            gold[_norm_q(r["question"])] = float(m.group(0).replace(",", ""))
    rows, missed = [], 0
    for rec in sg:
        q = " ".join(rec["input"].split())
        g = gold.get(_norm_q(q))
        if g is None:
            missed += 1
            continue
        plan = "\n".join(l.strip() for l in rec["output"].split("\n") if l.strip())
        rows.append({"question": q, "plan": plan, "answer": _fmt_answer(g)})
    os.makedirs(os.path.dirname(NT["data_path"]), exist_ok=True)
    with open(NT["data_path"], "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[data] joined {len(rows)} rows ({missed} unmatched) -> {NT['data_path']}")
    return rows


NT_ROWS = build_nothink_dataset()


# =============================================================================
# 5.2  MANUAL LABEL MASKING  (this is the core fix for BUG 2/3)
# =============================================================================
# Tokenise prefix and answer SEPARATELY, concatenate, and set labels to -100 on
# the prefix span. No response-marker string search, so nothing can silently
# mask the whole sequence. An assert proves each row has >=1 supervised token.
class MaskedDataset(torch.utils.data.Dataset):
    def __init__(self, rows, tokenizer, prefix_tmpl, max_len):
        self.samples = []
        eos_id = tokenizer.convert_tokens_to_ids(EOS_LITERAL)
        skipped = 0
        for r in rows:
            prefix = (prefix_tmpl.replace("{question}", r["question"])
                                 .replace("{plan}", r.get("plan", "")))
            ans = r["answer"]
            pref_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
            ans_ids = tokenizer(ans, add_special_tokens=False)["input_ids"] + [eos_id]

            ids = pref_ids + ans_ids
            if len(ids) > max_len:                 # truncate the PREFIX, never the answer
                overflow = len(ids) - max_len
                pref_ids = pref_ids[overflow:]
                ids = pref_ids + ans_ids
            labels = [-100] * len(pref_ids) + list(ans_ids)

            if all(l == -100 for l in labels):     # must never happen now
                skipped += 1
                continue
            self.samples.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids),
                "labels": labels,
            })
        assert self.samples, "every row masked out - masking is still broken"
        assert skipped == 0, f"{skipped} rows had no supervised token"
        # prove supervision exists
        sup = sum(1 for l in self.samples[0]["labels"] if l != -100)
        print(f"[mask] {len(self.samples)} rows | sample supervised tokens = {sup} "
              f"(answer + EOS) | prefix masked = "
              f"{len(self.samples[0]['labels']) - sup}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate(batch, pad_id):
    """Left-unpadded, right-padded batch. labels padded with -100."""
    maxlen = max(len(b["input_ids"]) for b in batch)
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        pad = maxlen - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad_id] * pad)
        out["attention_mask"].append(b["attention_mask"] + [0] * pad)
        out["labels"].append(b["labels"] + [-100] * pad)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


# =============================================================================
# 5.3  NAN GUARD
# =============================================================================
class NanGuard(torch.nn.Module):
    pass  # placeholder to keep namespace tidy


from transformers import TrainerCallback


class StopOnNan(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs and (math.isnan(logs["loss"]) or math.isinf(logs["loss"])):
            print(f"\n[nan] loss={logs['loss']} at step {state.global_step} - stopping. "
                  f"If this fires with force_fp32=True, lower learning_rate to 5e-5.")
            control.should_training_stop = True
        return control


# =============================================================================
# 5.4  TRAIN ONE MODEL
# =============================================================================
PAD_CANDIDATES = ["<|vision_pad|>", "<|fim_pad|>", "<|image_pad|>", "<|PAD_TOKEN|>"]


def _prep_tok(tok, model):
    eos_id = tok.convert_tokens_to_ids(EOS_LITERAL)
    tok.eos_token = EOS_LITERAL
    model.config.eos_token_id = eos_id
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.eos_token_id = eos_id
        model.generation_config.max_length = None
    if tok.pad_token_id is None or tok.pad_token_id == eos_id:
        v = tok.get_vocab()
        ch = next((t for t in PAD_CANDIDATES if t in v and v[t] != eos_id), None)
        if ch is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
            model.resize_token_embeddings(len(tok))
            ch = "<|pad|>"
        else:
            tok.pad_token = ch
    model.config.pad_token_id = tok.pad_token_id
    return tok


def train_model(name, rows, prefix_tmpl, init_from=None, lr=None) -> str:
    print("\n" + "=" * 78)
    print(f"TRAIN {name}  init={init_from or 'BASE'}  lr={lr or NT['learning_rate']}  "
          f"fp32={NT['force_fp32']}")
    print("=" * 78)
    gc.collect()
    torch.cuda.empty_cache()

    sources = [init_from] if init_from else list(NT["base_candidates"])
    model = tok = None
    for src in sources:
        try:
            model, tok = FastLanguageModel.from_pretrained(
                model_name=src, max_seq_length=NT["max_seq_length"],
                dtype=None, load_in_4bit=True)
            print(f"[load] {src}")
            break
        except Exception as exc:
            print(f"[load] {src} failed: {type(exc).__name__}: {str(exc)[:120]}")
    if model is None:
        raise RuntimeError(f"could not load {sources}")

    tok = _prep_tok(tok, model)

    if init_from is None:
        model = FastLanguageModel.get_peft_model(
            model, r=NT["lora_r"], lora_alpha=NT["lora_alpha"],
            lora_dropout=NT["lora_dropout"], bias="none",
            target_modules=NT["target_modules"],
            use_gradient_checkpointing="unsloth",
            random_state=NT["seed"])
    else:
        FastLanguageModel.for_training(model)

    # deterministic split
    import random
    idx = list(range(len(rows)))
    random.Random(NT["seed"]).shuffle(idx)
    n_val = max(1, int(len(rows) * NT["val_ratio"]))
    tr_rows = [rows[i] for i in idx[n_val:]]
    va_rows = [rows[i] for i in idx[:n_val]]

    train_ds = MaskedDataset(tr_rows, tok, prefix_tmpl, NT["max_seq_length"])
    val_ds = MaskedDataset(va_rows, tok, prefix_tmpl, NT["max_seq_length"])

    use_bf16 = torch.cuda.is_bf16_supported() and not NT["force_fp32"]
    use_fp16 = (not use_bf16) and (not NT["force_fp32"])
    out_dir = os.path.join(NT["out_root"], name)

    args = TrainingArguments(
        output_dir=os.path.join(out_dir, "ckpt"),
        per_device_train_batch_size=NT["micro_batch_size"],
        gradient_accumulation_steps=NT["grad_accum"],
        num_train_epochs=NT["epochs"],
        learning_rate=lr or NT["learning_rate"],
        lr_scheduler_type="cosine", warmup_ratio=0.03,
        weight_decay=0.01, max_grad_norm=0.3,          # tighter clip: nan insurance
        optim="adamw_8bit",
        fp16=use_fp16, bf16=use_bf16,                  # both False => fp32
        logging_steps=20, save_strategy="no",
        eval_strategy="epoch", seed=NT["seed"], report_to="none",
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=lambda b: collate(b, tok.pad_token_id),
        callbacks=[StopOnNan()],
    )

    t0 = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    result = trainer.train()
    mins = (time.time() - t0) / 60
    loss = float(getattr(result, "training_loss", float("nan")))
    print(f"[done] {mins:.1f} min | final_loss={loss:.4f} | "
          f"peak {torch.cuda.max_memory_reserved()/1024**3:.1f} GB")
    if math.isnan(loss):
        print("[WARN] final loss is nan - adapter is unusable. Lower LR and retry.")

    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "nothink_manifest.json"), "w") as fh:
        json.dump({"name": name, "init_from": init_from,
                   "lr": lr or NT["learning_rate"], "epochs": NT["epochs"],
                   "final_loss": loss, "train_minutes": round(mins, 1),
                   "prefix_template": prefix_tmpl}, fh, indent=2)
    try:
        dst = os.path.join(NT["drive_root"], "adapters", name)
        if os.path.ismount("/content/drive"):
            shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(out_dir, dst)
            print(f"[save] drive -> {dst}")
    except Exception as exc:
        print(f"[save] drive skip: {type(exc).__name__}")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return out_dir


# =============================================================================
# 5.5  RUN CURRICULUM
# =============================================================================
RUN_M1, RUN_M2, RUN_M3 = True, True, True
M1_DIR = os.path.join(NT["out_root"], "nothink_m1_scaffold")
M2_DIR = os.path.join(NT["out_root"], "nothink_m2_faded")
M3_DIR = os.path.join(NT["out_root"], "nothink_m3_control")

if RUN_M1:
    train_model("nothink_m1_scaffold", NT_ROWS, SCAFFOLD_PREFIX)
if RUN_M2:
    train_model("nothink_m2_faded", NT_ROWS, DIRECT_PREFIX,
                init_from=M1_DIR, lr=NT["lr_stage2"])
if RUN_M3:
    train_model("nothink_m3_control", NT_ROWS, DIRECT_PREFIX)


# =============================================================================
# 5.6  EVALUATE  (robust parser - fixes the first-number bug)
# =============================================================================
NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")
EQ_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")


def _to_f(t):
    try:
        return float(str(t).replace(",", "").replace("$", "").rstrip("."))
    except (ValueError, AttributeError):
        return None


def _match(a, b):
    return a is not None and b is not None and math.isclose(a, b, rel_tol=1e-4, abs_tol=1e-4)


def parse_answer(raw: str):
    """LAST equation result, then \\boxed, then last number. A terse model emits
    just a number; a still-verbose one puts the answer after the final '='."""
    if not raw:
        return None
    b = BOXED_RE.findall(raw)
    if b:
        v = _to_f(NUM_RE.search(b[-1]).group(0)) if NUM_RE.search(b[-1]) else None
        if v is not None:
            return v
    eq = EQ_RE.findall(raw)
    if eq:
        v = _to_f(eq[-1])
        if v is not None:
            return v
    nums = NUM_RE.findall(raw)
    return _to_f(nums[-1]) if nums else None


@torch.no_grad()
def _gen(model, tok, prompt, max_new_tokens):
    dev = next(model.parameters()).device
    inp = tok(prompt, return_tensors="pt", add_special_tokens=False).to(dev)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         eos_token_id=tok.convert_tokens_to_ids(EOS_LITERAL),
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def eval_model(adapter_dir, label, n=150, max_new_tokens=128):
    # 128 not 24: an under-trained model still writes CoT, and truncating it to
    # 24 tokens hides whether it reached the answer. A terse model stops early
    # via EOS anyway, so this does not slow the good case much.
    print(f"\n[eval] {label}")
    model, tok = FastLanguageModel.from_pretrained(
        model_name=adapter_dir, max_seq_length=NT["max_seq_length"],
        dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None

    test = load_dataset("openai/gsm8k", "main", split="test")
    correct, times, toks, samples = 0, [], [], []
    for i in range(min(n, len(test))):
        q = " ".join(test[i]["question"].split())
        gold = _to_f(test[i]["answer"].split("####")[-1].strip())
        prompt = DIRECT_PREFIX.replace("{question}", q)
        t0 = time.time()
        raw = _gen(model, tok, prompt, max_new_tokens)
        times.append(time.time() - t0)
        toks.append(len(tok(raw, add_special_tokens=False)["input_ids"]))
        pred = parse_answer(raw)
        correct += int(_match(pred, gold))
        if i < 6:
            samples.append((gold, pred, raw[:60]))
        if (i + 1) % 25 == 0:
            print(f"   {i+1}/{n} acc={correct/(i+1):.1%} {sum(times)/len(times):.2f}s "
                  f"{sum(toks)/len(toks):.0f}tok")
    acc = correct / min(n, len(test))
    print(f"[eval] {label}: {correct}/{min(n,len(test))} = {acc:.2%}  "
          f"{sum(times)/len(times):.2f}s  {sum(toks)/len(toks):.0f} tok/ans")
    for g, p, r in samples:
        print(f"     gold={g} pred={p} raw={r!r}")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {"label": label, "accuracy": acc, "correct": correct,
            "n": min(n, len(test)), "latency_s": sum(times)/len(times),
            "tokens": sum(toks)/len(toks)}


EVAL_N = 150
results_nt = []
for d, lab in [(M2_DIR, "M2 faded"), (M3_DIR, "M3 control"), (M1_DIR, "M1 scaffold (needs plan)")]:
    if lab.startswith("M1"):
        continue  # M1 needs a plan at inference; not a 1-SLM. Eval via 2-SLM path if wanted.
    if os.path.exists(os.path.join(d, "adapter_config.json")):
        results_nt.append(eval_model(d, lab, EVAL_N))


# =============================================================================
# 5.7  COMPARISON
# =============================================================================
TWO_SLM = [
    {"label": "Phase B alone (CoT)",     "accuracy": 0.6467, "latency_s": 16.05},
    {"label": "2-SLM guided (A+B)",      "accuracy": 0.6600, "latency_s": 26.69},
]
print("\n" + "=" * 74)
print(f"{'configuration':<30}{'acc':>8}{'latency':>10}{'speedup':>9}")
print("-" * 74)
ref = TWO_SLM[1]["latency_s"]
for r in TWO_SLM + results_nt:
    print(f"{r['label']:<30}{r['accuracy']:>7.1%}{r['latency_s']:>9.2f}s{ref/r['latency_s']:>8.1f}x")
print("-" * 74)
if len(results_nt) >= 2:
    d = results_nt[0]["accuracy"] - results_nt[1]["accuracy"]
    print(f"\nCURRICULUM EFFECT (M2 - M3): {d:+.2%}")
    print("Run McNemar before claiming it; <~8 points at n=150 is inside the noise band.")
print("""
Sanity gates before trusting ANY of this:
  - each model's final_loss must be a real number, not nan (check the [done] line)
  - [mask] must report supervised tokens > 0 and MANY rows (not 2)
  - M3 accuracy should be well above 0.7% now that it actually trained
If M2/M3 still land near 0%, direct-answer at 1.5B is simply very hard - that is
a real finding. But first confirm the three gates above passed.
""")

[data] cached 7473 rows

TRAIN nothink_m1_scaffold  init=BASE  lr=0.0001  fp32=True
==((====))==  Unsloth 2026.9.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[load] unsloth/Qwen2.5-Math-1.5B-bnb-4bit
[mask] 7100 rows | sample supervised tokens = 3 (answer + EOS) | prefix masked = 105


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[mask] 373 rows | sample supervised tokens = 5 (answer + EOS) | prefix masked = 243


...

[load] unsloth/Qwen2.5-Math-1.5B-bnb-4bit


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[mask] 7100 rows | sample supervised tokens = 3 (answer + EOS) | prefix masked = 40
[mask] 373 rows | sample supervised tokens = 5 (answer + EOS) | prefix masked = 82


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,100 | Num Epochs = 3 | Total steps = 1,332
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


Epoch,Training Loss,Validation Loss
1,0.742194,0.830371
2,0.570034,0.880146


---

## Reading the output

**`plan_has_math`** should sit near 0%. Anything above ~5% means Phase A is doing Phase B's job
and the separation-of-concerns premise is leaking. Fix it in the data, not the prompt.

**Answer-parse strategy mix.** A large `last_number` share means the `#### N` contract is being
ignored and you are effectively grading on "whatever number appeared last" — which inflates
accuracy on easy problems and is not defensible in a report. `hash` should dominate.

**`guided_only` vs `baseline_only`.** These are the two columns that matter for the write-up.
Net gain is `guided_only − baseline_only`; if `baseline_only` is large, the guidance is actively
misleading the executor on problems it could already solve, which is a more interesting finding
than the headline accuracy number.

---

## Suggested next steps

- **Ablate the executor.** Swap Phase B for Qwen2.5-0.5B and re-run the same slice. The base
  paper's finding was that an *untrained* responder beats a CoT-tuned one; confirming that at
  sub-1.5B scale is a genuine contribution.
- **Report latency honestly.** Cell 4 already prints the two-stage overhead multiple against
  single-stage. That number is the efficiency gap your Chapter 3 claims — put the measured value
  in, not an estimate.
- **Then attack it.** Guidance internalisation (the Curriculum Distillation "NoThink" stage) is
  the obvious follow-up: train Phase B on `question + plan → answer`, then drop the plan at
  inference and see how much of the gain survives.

In [1]:
   !zip -r /content/phaseA.zip /content/drive/MyDrive/SGFT_2SLM/adapters/phaseA_planner

	zip warning: name not matched: /content/drive/MyDrive/SGFT_2SLM/adapters/phaseA_planner

zip error: Nothing to do! (try: zip -r /content/phaseA.zip . -i /content/drive/MyDrive/SGFT_2SLM/adapters/phaseA_planner)
